In [ ]:
%run canvas/OEA_py_canvas

In [ ]:
import datetime
import json
from pyspark.sql import functions as F
from pyspark.sql.functions import col, avg, desc, first,asc,sum,when,substring, count, concat_ws, format_string, collect_list,last,round,monotonically_increasing_id,date_format,expr, regexp_replace,countDistinct,sha2, concat,col, split, expr,max
from scipy.stats import rankdata
from pyspark.sql.types import StringType
import time
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
import requests
import re

In [ ]:
class Schoology:
    def __init__(self, workspace='dev', version='0.1'):
        self.baseurl = "https://api.schoology.com/v1/"
        self.keyvault_consumer_key = f'schoologyConsumerKey{workspace}'
        self.keyvault_oauth_signature = f'schoologyOauthSignature{workspace}'
        self.version = version

    def set_workspace(self, workspace_name):
        oea.set_workspace(workspace_name)

    def set_version(self, version):
        self.version = version

    def auth_header(self):
        oauth_consumer_key = oea._get_secret(self.keyvault_consumer_key)
        oauth_token = ""
        # Generate dynamic values
        oauth_nonce = str(random.getrandbits(64))
        oauth_timestamp = int(time.time())

        # OAuth parameters
        oauth_signature_method = "PLAINTEXT"
        oauth_version = "1.0"


        # Construct the key for HMAC-SHA1

        # Calculate the signature using HMAC-SHA1
        oauth_signature = oea._get_secret(self.keyvault_oauth_signature)

        # Construct the Authorization header
        auth_header = (
        'OAuth realm="Schoology API", '
        f'oauth_consumer_key="{oauth_consumer_key}", '
        f'oauth_nonce="{oauth_nonce}", '
        f'oauth_timestamp="{oauth_timestamp}", '
        f'oauth_signature_method="{oauth_signature_method}", '
        f'oauth_version="{oauth_version}", '
        f'oauth_signature="{oauth_signature}"'
        )
        return auth_header

    def fetch_data(self, dataurl, resultName = 'user'):
        data_list = []
        while dataurl:
            header = self.auth_header()
            response = requests.get(dataurl, headers={"Authorization": header})
            if response.status_code == 200:
                data = response.json()
                data_list.extend(data[resultName])  # Adjust this based on the structure of your API response
                dataurl = data["links"].get("next", None) if "links" in data else None
            else:
                print(f"Error: {response.status_code}")
                break
        return data_list
    
    def load_users(self):
        currentDate = datetime.datetime.now()
        currentDateTime = currentDate.strftime("%Y-%m-%d")
        data = self.fetch_data(f'{self.baseurl}users')
        inactive_users = self.fetch_data(f'{self.baseurl}users/inactive')
        for user in inactive_users:
            user['name_display'] = user.pop('name')         # name_dispaly column to match with data 
        all_users = data + inactive_users
        # oea.land(json.dumps(all_users), f'schoology_raw/v{self.version}/users_inactive', 'inactive_users_data.json', oea.DELTA_BATCH_DATA, currentDateTime)
        oea.land(json.dumps(all_users), f'schoology_raw/v{self.version}/users', 'users.json', oea.DELTA_BATCH_DATA, currentDateTime)
       

    def load_roles(self):
        currentDate = datetime.datetime.now()
        currentDateTime = currentDate.strftime("%Y-%m-%d")
        data = self.fetch_data(f'{self.baseurl}roles', 'role')
        oea.land(json.dumps(data), f'schoology_raw/v{self.version}/roles', 'roles.json', oea.DELTA_BATCH_DATA, currentDateTime)
        

    def csv_files_list(self, path, details_list):
        try:
            items = mssparkutils.fs.ls(oea.to_prelanding_url(path))
            for item in items:
                if item.isFile and item.name.endswith('.csv'):
                    details_list.append([path, item.name])
                elif item.isDir:
                    self.csv_files_list(path + "/" + item.name, details_list)
        except Exception as e:
            logger.warning("[OEA] Could not get list of folders in specified path: " + path + "\nThis may be because the path does not exist.")
        


    # Call the function to gather details list
    def get_files_from_bulk(self, path):
        details_list = []
        self.csv_files_list(path,details_list)
        # Process the CSV files one by one
        for folder_path, filename in details_list:
            # print(folder_path, filename)
            self.preland_schoology_csv(folder_path, filename)
    
    #preland bulk data (Folders uploaded in Raw_Files)
    def preland_raw_folder(self):
        path = "oea/Raw_Files/Schoology"
        items = oea.get_folders(oea.to_prelanding_url(path))
        for item in items:
            if item!="Done_Prelanding":                     # To ensure already prelanded folders do not get triggered again 
                source_path = path+'/'+item
                self.get_files_from_bulk(source_path)      
                latest_dt = datetime.datetime.now().strftime("%Y-%m-%d")  
                destination_path = f'{path}/Done_Prelanding/{latest_dt}/'
                mssparkutils.fs.mkdirs(oea.to_prelanding_url(destination_path)) 
                mssparkutils.fs.mv(oea.to_prelanding_url(source_path),oea.to_prelanding_url(destination_path),overwrite=True)


    def preprocess_schoology_dataset(self, tables_source):
        """ Stage1/Transactional/Schoology_raw/{version}/{item}/oea.SNAPSHOT_BATCH_DATA/rundate=...   ==> Stage1/Transactional/Schoology/{version}/{item}/oea.SNAPSHOT_BATCH_DATA/rundate=...
        """
        from pyspark.sql.functions import when, col
        items = oea.get_folders(tables_source)
        for item in items:
            if item == '_preprocessed_tables':
                logger.info('Ignoring existing _preprocessed_tables folder.')
            else:
                table_path = tables_source +'/'+ item
                # find the batch data type of the table
                batch_type_folder = oea.get_folders(table_path)
                batch_type = batch_type_folder[0]
                # grab only the latest folder in stage1, used to write the JSON -> CSV to the same rundate folder timestamp
                # idea is to mimic the same directory structure of tables landed in stage1
                latest_dt = oea.get_latest_runtime(f'{table_path}/{batch_type}', "rundate=%Y-%m-%d")
                latest_dt = latest_dt.strftime("%Y-%m-%d")
                if item == 'users' or item == 'roles' or item == 'standards':
                    df = spark.read.json(oea.to_url(f'{table_path}/{batch_type}/rundate={latest_dt}/*.json'))
                else:
                    # df = pd.read_csv(oea.to_url(f'{table_path}/{batch_type}/rundate={latest_dt}/*.csv'))
                    # df = spark.read.csv(oea.to_url(f'{table_path}/{batch_type}/rundate={latest_dt}/*.csv'), header=True)
                    df = spark.read.option("quote", "\"").option("escape", "\"").option("encoding", "UTF-8").option("multiLine", True).csv(oea.to_url(f'{table_path}/{batch_type}/rundate={latest_dt}/*.csv'), header=True)

                if item == 'users':
                    df = df.withColumn('parents', df['parents'].cast(StringType()))
                elif item == 'roles':
                    df = df.withColumn('links', df['links'].cast(StringType())) 
                elif item == 'standards':
                    df = df.withColumn('description', regexp_replace(df['description'], r'\\r\\n', ' '))
                
                
                # if item == 'users':
                #     df['parents'] = df['parents'].astype(str)
                # elif item == 'roles':
                #     df['links'] = df['links'].astype(str)
                # elif item == 'standards':
                #     df['description'] = df['description'].str.replace(r'rn', ' ')
                    
                # ad hoc step(s) 
                #if item == 'accounts':
                #   df = df.withColumn('parent_account_id', df['parent_account_id'].cast(LongType()))
                #else:
                #    logger.info(f'no ad hoc processing needed for the Schoology {item} table.')
                # create the new location for the converted CSVs, and write back to stage1
                
                if item in ['question_data', 'student_submissions', 'submission_summary']:
                    # split_cols = df['File_Path'].str.split('/')
                    # df['Session'] = split_cols.str[0]
                    # df['Assessment_type'] = split_cols.str[1].str.split('-').str[1].str.strip()
                    # df['Subject'] = split_cols.str[2].str.split('-').str[1].str.strip()
                    # df['Grade'] = split_cols.str[3].str.split('-').str[1].str.strip()
                    # df['Section'] = split_cols.str[4].str.strip()
                    # df['File_Name'] = split_cols.str[5]
                    # df = df.drop(columns=['File_Path']) 
                    split_cols = F.split(F.col("File_Path"), '/')
                    assessment_type = F.when(
                        split_cols.getItem(1).contains('-'),
                        F.concat_ws("-", F.slice(F.split(split_cols.getItem(1), '-'), 2, F.size(F.split(split_cols.getItem(1), '-')) - 1))
                    ).otherwise(
                        F.trim(split_cols.getItem(1))
                    )
                    # split_assessment = F.split(split_cols.getItem(1), '-')
                    # assessment_type = F.concat_ws("-", F.slice(split_assessment, 2, F.size(split_assessment) - 1))
                    # assessment_type = F.trim(split_cols.getItem(1))
                    
                    df = df.withColumn("Session", split_cols.getItem(0)) \
                        .withColumn("Assessment_type", assessment_type) \
                        .withColumn("Subject", F.trim(split_cols.getItem(2))) \
                        .withColumn("Grade", F.trim(split_cols.getItem(3))) \
                        .withColumn("Section", F.trim(split_cols.getItem(4))) \
                        .withColumn("File_Name", split_cols.getItem(5)) \
                        .drop("File_Path")

                    df_tenant_base_school_id = oea.load(f"stage3/Published/schoology/v{schoology.version}/dim_school_tenant")
                    rows = df_tenant_base_school_id.select("School_ID", "Tenant_ID").collect()

                    subject_expr = F.col("Subject")
                    coursename_expr = F.col("Subject")
                    # subject overide

                    subject_expr = F.col("Subject")   # start with original value

                    for r in rows:
                        school_id = r.School_ID
                        tenant_id = r.Tenant_ID
                        
                        url = f"https://api.edvancelearning.us/Reporting/api/app/tenant-config/tenant-config/{tenant_id}?key=SubjectOverrideConfigJson"

                        try:
                            response = requests.get(url, timeout=10)
                            response.raise_for_status()
                            data1 = response.json()
                        except Exception as e:
                            print(f"Error for tenant_id {tenant_id}: {e}")
                            data1 = None

                        # Skip if no rules
                        if not data1:
                            continue

                        # Loop through subject override rules for this school
                        for rule in data1:
                            grade_val = rule.get("Grade")
                            subject_val = rule.get("Subject")
                            override_val = rule.get("SubjectOverride")

                            # Add School_ID condition also
                            subject_expr = (
                                F.when(
                                    (F.col("Grade") == grade_val) &
                                    (F.col("Subject") == subject_val) &
                                    (F.col("User_School_ID") == school_id),
                                    override_val
                                ).otherwise(subject_expr)
                            )
                    # Apply final expression to dataframe
                    if "User_School_ID" in df.columns:
                        df = df.withColumn("Subject", subject_expr)
                    ## For mapping the second case ##

                    coursename_expr = F.col("Subject")   # start with original value

                    for r in rows:
                        school_id = r.School_ID
                        tenant_id = r.Tenant_ID
                        
                        url2 = f"https://api.edvancelearning.us/Reporting/api/app/tenant-config/tenant-config/{tenant_id}?key=MapSubjectWithCourseNameJson"

                        try:
                            response = requests.get(url2, timeout=10)
                            response.raise_for_status()
                            data2 = response.json()
                        except Exception as e:
                            print(f"Error for tenant_id {tenant_id}: {e}")
                            data2 = None


                        # Skip if no rules
                        if not data2:
                            continue

                        # Loop through subject override rules for this school
                        for rule in data2:
                            grade_val = rule.get("Grade")
                            subject_val = rule.get("Subject")
                            regex_val = rule.get("CourseNameMatchRegex")
                            override_val = rule.get("SubjectOverride")

                            # Add School_ID condition also
                            coursename_expr = (
                                F.when(
                                    (F.col("Grade") == grade_val) &
                                    (F.col("Subject") == subject_val) &
                                    (F.col("Course_Name") == regex_val) &
                                    (F.col("User_School_ID") == school_id),
                                    override_val
                                ).otherwise(coursename_expr)
                            )
                    # Apply final expression to dataframe
                    if "User_School_ID" in df.columns:
                        df = df.withColumn("Subject", coursename_expr)
                    # # handling the subject names(Math-> Algebra for Grade 8 , and World History -> History for all Grades)
                    # if "User_School_ID" in df.columns:
                    #     df = df.withColumn(    
                    #             "Subject",
                    #             F.when((F.col("User_School_ID") == "186370968") & (F.col("Grade") == "Grade 8") & (F.col("Subject") == "Math"), "Algebra")
                    #             .when((F.col("User_School_ID") == "186370968") & (F.col("Grade") == "Grade 8") & (F.col("Subject") == "History"), "Civics")
                    #             .when((F.col("User_School_ID") == "186370968") & (F.col("Grade") == "Grade 6") & (F.col("Subject") == "Social Studies"), "History")
                    #             .when((F.col("User_School_ID") == "186370968") & (F.col("Grade") == "Grade 7") & (F.col("Subject") == "Social Studies"), "History")
                    #             .when((F.col("User_School_ID") == "186370968") & (F.col("Grade") == "Grade 8") & (F.col("Subject") == "Social Studies"), "Civics")
                    #             .when((F.col("User_School_ID") == "554425139") & (F.col("Grade") == "Grade 7") & (F.col("Subject") == "History"), "Civics")
                    #             .when((F.col("User_School_ID") == "554425139") & (F.col("Grade") == "Grade 8") & (F.col("Subject") == "Math"), "Algebra")
                    #             .when((F.col("User_School_ID") == "554425139") & (F.col("Grade") == "Grade 9") & (F.col("Subject") == "ELA"),"HS English I")
                    #             .when((F.col("User_School_ID") == "554425139") & (F.col("Grade") == "Grade 10") & (F.col("Subject") == "ELA"),"HS English II")
                    #             .when((F.col("User_School_ID") == "554425139") & (F.col("Grade") == "Grade 11") & (F.col("Subject") == "ELA"),"HS English III")
                    #             .when((F.col("User_School_ID") == "554425139") & (F.col("Grade") == "Grade 12") & (F.col("Subject") == "ELA"),"HS English IV")
                    #             .otherwise(F.col("Subject"))
                    #         )
                    #     df = df.withColumn("Grade", F.when((F.col("User_School_ID") == "554425139")& (F.col("Grade").isin(["Grade 9", "Grade 10", "Grade 11", "Grade 12"])),"Regular 9–12").otherwise(F.col("Grade")))
                    # # df = df.withColumn(
                    # #     "Assessment_type",
                    # #     F.when((F.col("Subject").isin("History", "Civics","World History","US History")), "Lesson Assessments")
                    # #     .otherwise(F.col("Assessment_type"))
                    # # )

                if item == 'question_data':
                    keep_null_standards = ['Standards', 'Standards17']
                    df = df.filter(
                        (F.col('Standards').isin(keep_null_standards)) | 
                        (F.col('Standards_Val').isNotNull()) | 
                        (F.col('Standards').isNull() & F.col('Standards_Val').isNull())
                    )
                    question_df = df
                    

                elif item == 'student_submissions':
                    teacher_dict_athenian = {
                        "Evan Markowitz, Jannette Rivera": "Jannette Rivera",
                        "Maria Baclohan, Evan Markowitz": "Maria Baclohan",
                        "Evan Markowitz, Mason Reeder": "Mason Reeder",
                        "Melaina Fijalkowski, Kathleen Tsakonas": "Kathleen Tsakonas",
                        "Evan Markowitz, Elizabeth Sedlak": "Elizabeth Sedlak",
                        "Daniel Smith, Nicole Swidarski": "Nicole Swidarski",
                        "Susanna Birdwell, Evan Markowitz": "Susanna Birdwell",
                        "Evan Markowitz, Niki Paul": "Niki Paul",
                        "Evan Markowitz, Tiffany Schaefer": "Tiffany Schaefer",
                        "Evan Markowitz, Kathleen Tsakonas": "Kathleen Tsakonas",
                        "Susanna Birdwell, Melaina Fijalkowski": "Susanna Birdwell",
                        "Daniel Smith, Heather Watkins": "Heather Watkins",
                        "Evan Markowitz, Mary Sidhom": "Mary Sidhom",
                        "Carissa Farrell, Madison Wahn": "Carissa Farrell",
                        "Mary Sidhom, Madison Wahn": "Madison Wahn",
                        "Evan Markowitz, Pattie Rossi": "Pattie Rossi",
                        "Gabriela Agostino, Sitara Qalander": "Gabriela Agostino",
                        "Sharon Long, Pattie Rossi": "Pattie Rossi",
                        "Ana Leiva, Mary Vaughn": "Mary Vaughn",
                        "Evan Markowitz, Sitara Qalander, Jannette Rivera": "Jannette Rivera",
                        "Maria Baclohan, Sharon Long": "Maria Baclohan",
                        "Gabriela Agostino, Evan Markowitz": "Gabriela Agostino",
                        "Ashley Lekhram, Evan Markowitz": "Ashley Lekhram",
                        "Ana Leiva, Evan Markowitz, Mary Vaughn": "Ana Leiva",
                        "Evan Markowitz, Nicole Swidarski": "Nicole Swidarski",
                        "Carissa Farrell, Evan Markowitz": "Carissa Farrell",
                        "Elizabeth Bennet, Melaina Fijalkowski": "Elizabeth Bennet",
                        "Melaina Fijalkowski, Nicole Swidarski": "Nicole Swidarski",
                        "Maria Baclohan, Sitara Qalander": "Maria Baclohan",
                        "Taylor Almendinger, Mason Reeder, Elizabeth Sedlak": "Taylor Almendinger",
                        "Sitara Qalander, Elizabeth Sedlak": "Elizabeth Sedlak"
                    }
                    teacher_dict_brightview = {
                        "Heather Fernandez, Rommy Rodriguez, Arian Rubio": "Arian Rubio",
                        "Rommy Rodriguez, Arian Rubio": "Arian Rubio",
                        "Elizabeth McKinney, Rommy Rodriguez": "Elizabeth McKinney",
                        "Kenia Gomez, Rommy Rodriguez": "Kenia Gomez",
                        "Elizabeth McKinney, Melba Montano, Rommy Rodriguez": "Melba Montano",
                        "Kenia Gomez, Jannette Rivera, Rommy Rodriguez": "Kenia Gomez",
                        "Dayanis Ceballo, Rommy Rodriguez": "Dayanis Ceballo",
                        "Melba Montano, Rommy Rodriguez": "Melba Montano"
                    }
                    teacher_dict_southprep = {
                        "Dayanis Ceballo, Rommy Rodriguez": "Dayanis Ceballo"
                    }
                    from pyspark.sql.functions import col, when,udf

                    def replace_values(value, school_id):
                        if school_id == '186370968':
                            teacher_dict = teacher_dict_athenian
                        elif school_id == '7448280461':
                            teacher_dict = teacher_dict_brightview
                        elif school_id == '7368546879':
                            teacher_dict = teacher_dict_southprep
                        else:
                            return value  

                        return teacher_dict.get(value, value)
                    # Define the UDF using the replace_values function
                    replace_values_udf = udf(lambda value, school_id: replace_values(value, school_id), StringType())

                    # Apply the UDF to the DataFrame
                    df = df.withColumn(
                        "Section_Instructors_values",
                        when(
                            col("Section_Instructors").contains(","),
                            replace_values_udf(col("Section_Instructors"), col("User_School_ID"))
                        ).otherwise(col("Section_Instructors"))
                    )
                    df = df.drop("Section_Instructors").withColumnRenamed("Section_Instructors_values", "Section_Instructors")
                    student_submissions_df = df
            
                elif item == 'submission_summary':
                    
                    df = df.withColumnRenamed('Schoology_ID', 'User_ID')
                    df = df.withColumn('question_No', split(df['Question'], '_').getItem(1))
                    df = df.drop("Question")
                    df = df.withColumn('File_Name_Item_removed', regexp_replace(df['File_Name'], 'Submission-Summary-', ''))
                    question_df = question_df.withColumn('File_Name_QItem_removed', regexp_replace(question_df['File_Name'], 'Question-Data-', ''))
                    merged_df = df.join(question_df.select('Item_ID', 'File_Name_QItem_removed', 'Question_No', 'Question_ID').dropDuplicates(['File_Name_QItem_removed', 'Question_No']),
                            (df['File_Name_Item_removed'] == question_df['File_Name_QItem_removed']) & 
                            (df['question_No'] == question_df['Question_No']),
                            'left')
                    df = merged_df.withColumn('Question_ID', col('Question_ID')).withColumn('Item_ID', col('Item_ID'))
                    merged_df2 = df.join(student_submissions_df.select('User_UID', 'User_School_ID').dropDuplicates(['User_UID']),
                        df['User_ID'] == student_submissions_df['User_UID'],
                        'left')
                    df = merged_df2.withColumn('School_ID', col('User_School_ID')).drop('User_UID', 'User_School_ID')
                    df = df.drop('File_Name_QItem_removed','Question_No')
                  
            
                entity_path = f'schoology/v{self.version}/{item}'
                oea.land(df, entity_path = entity_path , rundate = f'{latest_dt}')
                # remove the _SUCCESS file
                new_table_path = f'stage1/Transactional/schoology/v{self.version}/{item}/{batch_type}/rundate={latest_dt}'
                oea.rm_if_exists(new_table_path + '/_SUCCESS', False)
            logger.info('Finished pre-processing Schoology tables')
        

    def preland_batch_comp_files(self,bcfolderPath,bcfileName):
        # will open batchCompleted csv file and iterate each row as file name and send to function preland_schoology_csv
        batches_txt_file_path = oea.to_prelanding_url(bcfolderPath + '/' + bcfileName)
        rdd = spark.sparkContext.textFile(batches_txt_file_path)
        for line in rdd.collect():
            if line.strip():  # Check for non-empty lines
                folder_path, file_name = line.split(',')
                full_folder_path = f"oea/pre_landing/Schoology/{folder_path.strip()}"
                try:                        #try statement if file path not found
                    self.preland_schoology_csv(full_folder_path, file_name.strip())
                except AnalysisException as e:
                    logger.info(f'File not found: {full_folder_path}/{file_name.strip()} - Exception: {e}')

        

    def preland_schoology_csv(self, folderPath, fileName):
        """ pre-landing/Schoology/{tenant}/Year....   ==> Stage1/Transactional/Schoology_raw/{version}/{item}/oea.SNAPSHOT_BATCH_DATA/rundate=...
        """
        csv_file_path=oea.to_prelanding_url(folderPath+'/'+fileName)
        # df=spark.read.format("csv").option("header","true").load(csv_file_path)
        expected_columns = ["Item ID","Item Name","Question ID","Associated Question ID","Total Points","Question Type","Question","Position Number","Sub-Question","Answer Option","Answer Breakdown","Answer Breakdown","Correct Answer","Correctly Answered","Most Points Earned","Least Points Earned","Average Points Earned","Standards"]

        df = spark.read.format("csv").option("header", "true") .option("quote", "\"").option("escape", "\"").option("quoteAll", "true").option("encoding", "ISO-8859-1").load(csv_file_path)
        if "Question-Data" in fileName:
            for col in expected_columns:
                if col not in df.columns:
                    df = df.withColumn(col, lit(None).cast(StringType()))

        df1 = oea.fix_column_names(df)
        
        if 'Question' in df1.columns:
            # Truncate 'Question' column to 7500 characters
            df1 = df1.withColumn("Question", expr("SUBSTRING(Question, 1, 7500)"))

        if 'Answer_Submission' in df1.columns:
            # Truncate 'Question' column to 7500 characters
            df1 = df1.withColumn("Answer_Submission", expr("SUBSTRING(Answer_Submission, 1, 7500)"))    


        pd_df = df1.toPandas()
        item = ''

        if "Question-Data" in fileName:
            item = 'question_data'
            list_all_cols = list(pd_df.columns)

            list_val_vars_Stan=[]
            [list_val_vars_Stan.append(w) if w.startswith("Standards") else "" for w in list_all_cols]

            list_id_var1 = []
            list_id_var1 = list(set(list_all_cols) - set(list_val_vars_Stan))


            if not list_val_vars_Stan:
                pd_df["Standards"] = None
                list_val_vars_Stan = ["Standards"]


            if not list_val_vars_Stan:
                pd_df["Standards"] = None
                list_val_vars_Stan = ["Standards"]

            # Melt on Standards
            df_melt1 = pd_df.melt(
                id_vars=list_id_var1,
                value_vars=list_val_vars_Stan,
                var_name='Standards',
                value_name='Standards_Val'
            )

            # Remove empty Standards_Val only if there are multiple rows for that Question_ID
            # standards_group_counts = df_melt1.groupby('Question_ID')['Standards_Val'].transform('count')
            # df_melt1 = df_melt1[
            #     ~((df_melt1['Standards_Val'].isna() | (df_melt1['Standards_Val'] == '')) & (standards_group_counts > 1))
            # ]

            standards_group_counts = df_melt1.groupby('Question_ID')['Standards_Val'].transform(lambda x: x.notna().sum())

            df_melt1 = df_melt1[
                ~((df_melt1['Standards_Val'].isna() | (df_melt1['Standards_Val'] == '')) & (standards_group_counts > 0))
            ]

            # Melt on Answer_Breakdown if applicable
            list_all_cols_melt1 = df_melt1.columns.tolist()
            list_val_vars_AnsB = [w for w in list_all_cols_melt1 if w.startswith("Answer_Breakdown")]

            if list_val_vars_AnsB:
                list_id_var2 = list(set(list_all_cols_melt1) - set(list_val_vars_AnsB))
                df_final = df_melt1.melt(
                    id_vars=list_id_var2,
                    value_vars=list_val_vars_AnsB,
                    var_name='Answer_Breakdown',
                    value_name='Answer_Breakdown_Val'
                )
            else:
                df_final = df_melt1.copy()

            # Ensure presence of expected columns
            for col in ['Standards', 'Standards_Val', 'Answer_Breakdown', 'Answer_Breakdown_Val']:
                if col not in df_final.columns:
                    df_final[col] = ''

            df_final['Unique_Key'] = (
                    df_final['Item_ID'].astype(str) +
                    df_final['Question_ID'].astype(str) +
                    df_final['Correct_Answer'].astype(str) +
                    df_final['Position_Number'].astype(str) +
                    df_final['Answer_Option'].astype(str) +
                    df_final['Answer_Breakdown'].astype(str) +
                    df_final['Standards'].astype(str)
            )

            df_final = df_final.sort_values(by="Question_ID").reset_index(drop=True)
            df_final['Question_No'] = rankdata(df_final['Question_ID'], method='dense').astype(int)

            df_final['File_Path'] = '/'.join(folderPath.split('/')[3:]) + fileName 
 
            
        elif "Submission-Summary" in fileName:
            item = 'submission_summary'
            list_all_cols = list(pd_df.columns)

            list_val_vars_Ques=[]
            [list_val_vars_Ques.append(w) if w.startswith("Question") else "" for w in list_all_cols]
            
            list_id_var1 = []
            list_id_var1 = list(set(list_all_cols) - set(list_val_vars_Ques))
            
            df_final = pd_df.melt(id_vars=list_id_var1, 
                        value_vars=list_val_vars_Ques,
                        var_name='Question', value_name='Question_Val')
            df_final['Unique_Key'] = (df_final['Schoology_ID'].astype(str) + df_final['Question'].astype(str))
            df_final['File_Path'] = '/'.join(folderPath.split('/')[3:]) + fileName  


        elif "Student-Submissions" in fileName:
            item = 'student_submissions'
            df_final = pd_df.copy()
            df_final['Unique_Key'] = (
                                    df_final['User_UID'].astype(str) +
                                    df_final['Item_ID'].astype(str) +
                                    df_final['Question_ID'].astype(str) +
                                    df_final['Position_Number'].astype(str) +
                                    df_final['Answer_Submission'].astype(str) +
                                    df_final['Points_Received'].astype(str) +
                                    df_final['Points_Possible'].astype(str) +
                                    df_final['Submission'].astype(str) +
                                    df_final['Correct_Answer'].astype(str)
                                )
            df_final['File_Path'] = '/'.join(folderPath.split('/')[3:]) + fileName

        elif "Attendance" in fileName:
            item = 'attendance'
            df_final = pd_df.copy()

        
        elif "SIS" in fileName:
            item = 'student_information'
            df_final = pd_df.copy()
        
        destinationFolderPPath = f'stage1/Transactional/schoology_raw/v{self.version}/{item}'
        
        # Use the current date and time
        latest_dt = datetime.datetime.now().strftime("%Y-%m-%d")        
        new_table_path = f'stage1/Transactional/schoology_raw/v{self.version}/{item}/{oea.DELTA_BATCH_DATA}/rundate={latest_dt}'

        #making directories using dbutils for pandas csv file writting
        mssparkutils.fs.mkdirs(oea.to_url(new_table_path))
        df_final.to_csv(oea.to_url(f'{new_table_path}/{fileName}'))

        # spark_df = spark.createDataFrame(df_final)
        # spark_df.coalesce(1).write.save(oea.to_url(f'{new_table_path}'), format='csv', mode='overwrite', header='true', mergeSchema='true')
    
    def process(self, source_path, foreach_batch_function, options={}):
        if not oea.path_exists(source_path):
            raise ValueError(f'The given path does not exist: {source_path} (which resolves to: {oea.to_url(source_path)})') 
        def wrapped_function(df, batch_id):
            df.persist() # cache the df so it doesn't get read in multiple times when we write to multiple destinations. See: https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#foreachbatch
            foreach_batch_function(df, batch_id)
            df.unpersist()
        spark.sql("set spark.sql.streaming.schemaInference=true")
        print(f"source_path is: {source_path}")
        streaming_df = spark.readStream.load(oea.to_url(source_path), **options)
        query = streaming_df.writeStream.format('delta').outputMode('append').trigger(once=True).option('checkpointLocation', oea.to_url(source_path) + '/_del_checkpoints').foreachBatch(wrapped_function).start()
        query.awaitTermination()   # block until query is terminated, with stop() or with error; A StreamingQueryException will be thrown if an exception occurs.
        # number_of_new_inbound_rows = query.lastProgress["numInputRows"]
        # logger.info(f'Number of new inbound rows processed: {number_of_new_inbound_rows}')
        # logger.debug(query.lastProgress)
        # return number_of_new_inbound_rows
    
    def ingest_to_df(self, entity_path, primary_key='id', options={}):
        """ mimicing the oea ingest to perfom deletion function on df
        """
        # primary_key = oea.fix_column_name(primary_key) 
        ingested_path = f'stage2/Ingested/{entity_path}'
        raw_path = f'stage1/Transactional/{entity_path}'
        if not oea.path_exists(raw_path):
            logger.error(f'Failed to ingest data because the given source data was not found where expected: {raw_path}')
            return
        batches = oea.get_batch_info(raw_path)
        # number_of_inbound_changes = 0
        for batch in batches:
            batch_type = batch[0]  
            print("batch_type",batch_type)          #delta
            source_data_format = batch[1]    # parquet
            logger.info(f'Ingesting from: {raw_path}, batch type of: {batch_type}, source data format of: {source_data_format}')
            source_url = oea.to_url(f'{raw_path}/{batch_type}_batch_data')
            if oea.get_folder_size(f'{source_url}/{oea.get_latest_folder(source_url)}') > 0:
                if batch_type == 'delta':
                    def batch_func(df, batch_id): 
                        # params = {'req_cols': ['Item_ID','Question_ID', 'Standards_Val']}
                        params = {
                        'req_cols': [
                            ['Item_ID','Question_ID', 'Standards_Val'],
                            ['Item_ID', 'Question_ID', 'Correct_Answer']
                        ]
                    }
                        req_cols_list = params.get('req_cols', [])
                        for req_cols in req_cols_list:
                            self.delete_from_path(df,ingested_path,req_cols)
                        # oea.upsert(df, ingested_path, primary_key)
                else:
                    raise ValueError("No valid batch folder was found at that path (expected to find a single folder with one of the following names: snapshot_batch_data, additive_batch_data, or delta_batch_data). Are you sure you have the right path?")                      
                if options == None: options = {}
                options['format'] = source_data_format # 'parquet' in our cas
                self.process(source_url, batch_func, options)
        #         if number_of_new_inbound_rows > 0:    
        #             self.add_to_lake_db(ingested_path)
        #         number_of_inbound_changes += number_of_new_inbound_rows
        # return number_of_inbound_changes

    def ingest_schoology_dataset(self,tables_source):
        """ Stage1/Transactional/canvas/{version}/{item}/oea.SNAPSHOT_BATCH_DATA/rundate=...  ==> Stage2/Ingested/canvas/{version}/{item}
        """
        items = oea.get_folders(f'stage1/Transactional/{tables_source}')
        for item in items:
            table_path = f'canvas/v{self.version}/{item}'
            print(table_path)
            try:
                # 3 paths: check_path is for checking whether the table should be ingested, read_path is for reading the stage1 CSV location, write path for stage2 ingested location
                if item == 'metadata.csv':
                    logger.info('ignore metadata csv - not a table to be ingested')
                elif item == 'users':
                    oea.ingest(table_path, 'uid')
                elif item == 'roles':
                    oea.ingest(table_path, 'id')
                elif item == 'standards':
                    oea.ingest(table_path,'Identifier')
                elif item == "student_information" or item =="attendance":
                    oea.ingest(table_path,'Student_ID')
                else:
                    if item == 'question_data':
                        self.ingest_to_df(table_path, "Unique_Key")
                    oea.ingest(table_path, "Unique_Key")
            except AnalysisException as e:
                # This means the table may have not been properly refined due to errors with the primary key not aligning with columns expected in the lookup table.
                pass
        logger.info('Finished ingesting the most recent Schoology data')

    def refine_schoology_dataset(tables_source):
            items = oea.get_folders(tables_source)
            for item in items: 
                table_path = tables_source +'/'+ item
                if item == 'metadata.csv':
                    logger.info('ignore metadata processing, since this is not a table to be ingested')
                else:
                    try:
                        pass
                        """ TODO: No refinement currently, will do later
                        """
                    except AnalysisException as e:
                        # This means the table may have not been properly refined due to errors with the primary key not aligning with columns expected in the lookup table.
                        pass
                    
                    logger.info('Refined table: ' + item + ' from: ' + table_path)
            logger.info('Finished refining Canvas tables')

    def delete_stale_rows(self, source_df, destination_path2, destination_path3, params):
        # req_cols = params.get('req_cols', ['School_ID','Question_ID', 'Standards'])
        req_cols_list = params.get('req_cols', [])
        for req_cols in req_cols_list:
            self.delete_from_path(source_df, destination_path2, req_cols)
            self.delete_from_path(source_df, destination_path3, req_cols)
    
    def delete_from_path(self, source_df, destination_path, req_cols):
        """Deletes rows from the destination Delta table where the Standards are present in the older data but not in the new data 
        for matching Question_ID.
        """
        if not req_cols:
            raise ValueError("req_cols is empty or None")
        logger.info(f"req_cols: {req_cols}")
        destination_url = oea.to_url(destination_path)
        if not DeltaTable.isDeltaTable(spark, destination_url):
            logger.debug("Delta table does not exist. Skipping deletion.")
            return
        delta_table_sink = DeltaTable.forPath(spark, destination_url)
        existing_df = delta_table_sink.toDF()
        # only incoming Question_IDs will be compared
        distinct_source_df = source_df.select(req_cols[0], req_cols[1]).distinct()
        filtered_existing_df = existing_df.join(distinct_source_df, on=req_cols[:2], how='inner')
        # on the basis of two columns (Question_ID , Standards), data will be deleted
        source_cols = source_df.select(*req_cols)
        rows_to_delete = filtered_existing_df.select(*req_cols).subtract(source_cols)
        logger.info("rows_to_delete: \n" + str(rows_to_delete.collect()))
        if rows_to_delete.count() > 0:
            delta_table_sink.alias('sink').merge(
                rows_to_delete.alias('updates'),
                f"""
                sink.{req_cols[0]} = updates.{req_cols[0]} AND
                sink.{req_cols[1]} = updates.{req_cols[1]} AND
                (sink.{req_cols[2]} = updates.{req_cols[2]} OR
                (sink.{req_cols[2]} IS NULL AND updates.{req_cols[2]} IS NULL))
                """).whenMatchedDelete().execute()
        else:
            logger.debug("No rows to delete.")


    def _publish_to_stage2(self,df, destination, pk):
        oea.upsert(df, destination, pk)
    def _publish_to_stage3(self,df, destination, pk):
        oea.upsert(df, destination, pk)
    
    def publish(self, df, stage2_destination, stage3_destination, primary_key='id'):
        existing_count = 0
        try:      # if table not found (First Table)
            existing_df = spark.read.format('delta').load(oea.to_url(stage2_destination))
            existing_count = existing_df.count()
        except AnalysisException as e:
            # If the table does not exist, continue without logging anything
            if 'Path does not exist' in str(e):
                pass
            else:
                # Raise the exception if it's not related to the path not existing
                raise e
        
        self._publish_to_stage2(df, stage2_destination, primary_key)
        self._publish_to_stage3(df, stage3_destination, primary_key)
        streaming_df = spark.read.format('delta').load(oea.to_url(stage2_destination))
        new_count = streaming_df.count()
        # Calculate the number of new inbound rows
        number_of_new_inbound_rows = new_count - existing_count
        logger.info(f'Number of new inbound rows processed: {number_of_new_inbound_rows}')
        return number_of_new_inbound_rows                       
    

    
    def build_dimension_tables(self,tables_source):
        """ Stage2/Ingested/canvas/{version}/{item}  ==> Stage2/Enriched/canvas/{version}/{item}
        """
        def fix_urls(question):
            pattern = r"http://app\.schoology\.com/system/files/.*?/(attachments/page_embeds)"
            replacement = r"http://app.schoology.com/system/files/\1"
            cleaned_question = re.sub(pattern, replacement, question)
            return cleaned_question

        # Define the UDF for cleaning URLs in the Question column
        fix_urls_udf = F.udf(fix_urls, StringType())

        # df_User = oea.load(f'stage2/Ingested/canvas/v{self.version}/users')
        df_Question = oea.load(f'stage2/Ingested/canvas/v{self.version}/question_data')
        df_Question = df_Question.withColumn('Question', fix_urls_udf(F.col('Question')))
        keep_null_standards = ['Standards', 'Standards17']
        df_Question = df_Question.filter(
              (F.col('Standards').isin(keep_null_standards)) | 
            (F.col('Standards_Val').isNotNull()) | 
            (F.col('Standards').isNull() & F.col('Standards_Val').isNull())
        )
        df_Student_Submissions = oea.load(f'stage2/Ingested/canvas/v{self.version}/student_submissions')
        # df_Submission_Summary = oea.load(f'stage2/Ingested/canvas/v{self.version}/submission_summary')
        # df_Student_Info = oea.load(f'stage2/Ingested/schoology/v{self.version}/student_information')
         # genearting unique id for (dim_grade,dim_assessment_type, dim_subject) after student submissions fact table
        def generate_uuid_2(column1, column2):
            combined_column = concat_ws("_",column1, column2)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid

        def generate_uuid_5(column1, column2,column3 ,column4, column5):
            combined_column = concat_ws("_",column1,column2,column3,column4,column5)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid
        def generate_uuid_6(column1, column2,column3 ,column4, column5, column6):
            combined_column = concat_ws("_",column1,column2,column3,column4,column5,column6)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid
        def generate_uuid_7(column1, column2,column3 ,column4, column5, column6, column7):
            combined_column = concat_ws("_",column1,column2,column3,column4,column5,column6,column7)
            combined_column = concat(column1,column2,column3,column4,column5,column6,column7)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid

        def generate_uuid_8(column1, column2,column3 ,column4, column5, column6, column7,column8):
            combined_column = concat_ws("_",column1,column2,column3,column4,column5,column6,column7,column8)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid      
    
        df_Student_Submissions = df_Student_Submissions.withColumnRenamed('User_School_ID','School_ID').withColumnRenamed('User_School_Name','School_Name')
        # df_User = df_User.withColumnRenamed('school_id','School_ID')
        
        # dim_school
        df_dim_school = df_Student_Submissions[['School_ID','School_Name']].drop_duplicates().dropna(subset=['School_ID', 'School_Name'])
        self.publish(df_dim_school, f'stage2/Enriched/canvas/v{self.version}/dim_school',f'stage3/Published/canvas/v{self.version}/dim_school', primary_key='School_ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_school')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_school')
        

        # Dim_Role
        # df_Dim_Role = 
        # Role_ID	    numeric
        # Role_Name	nvarchar(max)

        # # dim_student
        # df_dim_student = df_User.filter(df_User['role_id'] == 286170)
        # df_dim_student= df_dim_student[['uid','id','School_ID','school_uid','name_title','name_first','name_first_preferred',
        # 'use_preferred_first_name','name_middle','name_middle_show','name_last','name_display','primary_email','picture_url',
        # 'gender','position','grad_year','username','password','role_id','tz_offset','tz_name','language']].drop_duplicates()

        # df_dim_stu_info = df_Student_Info.select("Student_ID", "Grade", "Home_Room_Teacher", "English_Language_Learner_ELL_", "Primary_Exceptionality", "Race")
        # Joining the dataframes
        # joined_df = df_dim_student.join(df_dim_stu_info,df_dim_stu_info["Student_ID"] == df_dim_student["school_uid"],"left")
        # joined_df = joined_df.drop("school_uid")
        # self.publish(df_dim_student, f'stage2/Enriched/canvas/v{self.version}/dim_student',f'stage3/Published/canvas/v{self.version}/dim_student', primary_key='uid')
        # oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_student')
        # oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_student')


        # # dim_teacher
        # df_dim_teacher = df_User.filter(df_User['role_id'] == 286168)
        # df_dim_teacher= df_dim_teacher[['uid','id','School_ID','name_title','name_first','name_first_preferred',
        # 'use_preferred_first_name','name_middle','name_middle_show','name_last','name_display','primary_email','picture_url',
        # 'gender','position','grad_year','username','password','role_id','tz_offset','tz_name','language']].drop_duplicates()
        # self.publish(df_dim_teacher, f'stage2/Enriched/canvas/v{self.version}/dim_teacher',f'stage3/Published/canvas/v{self.version}/dim_teacher', primary_key='uid')
        # oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_teacher')
        # oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_teacher')


        # # dim_parent
        # df_dim_parent = df_User.filter(df_User['role_id'] == 286172)
        # df_dim_parent= df_dim_parent[['uid','id','School_ID','name_title','name_first','name_first_preferred',
        # 'use_preferred_first_name','child_uids','name_middle','name_middle_show','name_last','name_display','primary_email','picture_url',
        # 'gender','position','grad_year','username','password','role_id','tz_offset','tz_name','language']].drop_duplicates()
        # self.publish(df_dim_parent, f'stage2/Enriched/canvas/v{self.version}/dim_parent',f'stage3/Published/canvas/v{self.version}/dim_parent', primary_key='uid')
        # oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_parent')
        # oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_parent')


        

        # dim_course
        df_dim_course = df_Student_Submissions[['Course_NID','Course_Name','Course_code','School_ID']].drop_duplicates().dropna(subset=['Course_NID','Course_Name','Course_code','School_ID'])
        self.publish(df_dim_course, f'stage2/Enriched/canvas/v{self.version}/dim_course',f'stage3/Published/canvas/v{self.version}/dim_course', primary_key='Course_NID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_course')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_course')

        # first fetching student submissions table as to get rows for 3 dim tables with unique ids (Grade_ID,Assessment_ID,Subject_ID)
        df_Student_Submissions = oea.load(f'stage2/Ingested/canvas/v{self.version}/student_submissions')
        df_Student_Submissions = df_Student_Submissions.withColumnRenamed('User_School_ID','School_ID').withColumnRenamed('User_School_Name','School_Name')
        df_Student_Submissions = df_Student_Submissions[['User_UID','User_Role_ID','School_ID','Course_NID','Section_NID',
        'Section_Code','Section_Name','Section_Instructors','Item_ID','Item_Type','Item_Name','First_Access','Latest_Attempt','Total_Time','Submission_Grade','Submission','Question_ID',
        'Session','Assessment_type','Subject','Grade','Section','File_Name',
        'Position_Number','Sub-Question','Answer_Submission','Correct_Answer','Points_Received','Points_Possible']]
        #adding one new column to be a primary key
        df_Student_Submissions = df_Student_Submissions.withColumn("User_id_ques_id",concat(col("User_UID"), lit('-'), col("Question_ID")))
        df_Student_Submissions = df_Student_Submissions.withColumn('Qkey', 
            concat(col('Session'), col('Assessment_type'), col('Subject'), col('Grade'), col('Question_ID'))
        )     
        # Selecting required columns 
        fact_Student_Submissions = df_Student_Submissions.select(
            'User_UID','User_Role_ID','School_ID','Course_NID','Section_NID','Section_Name','Section_Instructors',
        'Section_Code','Item_ID','Item_Type','Item_Name','First_Access','Latest_Attempt','Total_Time','Submission_Grade','Submission','Question_ID',
        'Session','Assessment_type','Subject','Grade','Section','File_Name',
        'Position_Number','Sub-Question','Answer_Submission','Correct_Answer','Points_Received','Points_Possible','User_id_ques_id'
        ).withColumn(
            "assessment_date", date_format(col("Latest_Attempt"), "MM/dd/yyyy")
        )
        fact_Student_Submissions = fact_Student_Submissions.withColumn("Grade_ID", generate_uuid_2(df_Student_Submissions['School_ID'], df_Student_Submissions['Grade']))
        fact_Student_Submissions = fact_Student_Submissions.withColumn("Assessment_ID", generate_uuid_2(df_Student_Submissions['School_ID'], df_Student_Submissions['Assessment_type']))
        fact_Student_Submissions = fact_Student_Submissions.withColumn("Subject_ID", generate_uuid_6(df_Student_Submissions['School_ID'], df_Student_Submissions['Subject'], df_Student_Submissions['Assessment_type'], df_Student_Submissions['Grade'], df_Student_Submissions['Session'],df_Student_Submissions['Item_Name']))
        
        # # dim_item  
        df_dim_item = fact_Student_Submissions[['Item_ID','Subject_ID','assessment_date','Section_Name','Section_Instructors','Item_Type','Item_Name','School_ID']].drop_duplicates()
        df_dim_item = df_dim_item[['Item_ID','Subject_ID','assessment_date','Section_Name','Section_Instructors','Item_Type','Item_Name','School_ID']].drop_duplicates().drop_duplicates().dropna(subset=['Item_ID', 'Subject_ID', 'Section_Name', 'Section_Instructors', 'Item_Type', 'Item_Name', 'School_ID','assessment_date'])
        # Define a window partitioned by unique columns, and order by assessment_date
        window_spec = Window.partitionBy(
            'Item_ID', 'Subject_ID', 'Section_Name', 'Section_Instructors', 'Item_Type', 'Item_Name', 'School_ID'
        ).orderBy(col('assessment_date').asc())
        # Add a row number based on the ordering within each group
        df_dim_item = df_dim_item.withColumn('row_num', row_number().over(window_spec))
        # Select only the first row from each group
        df_dim_item = df_dim_item.filter(col('row_num') == 1).drop('row_num')
        self.publish(df_dim_item, f'stage2/Enriched/canvas/v{self.version}/dim_item',f'stage3/Published/canvas/v{self.version}/dim_item', primary_key='Item_ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_item')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_item')

      # Getting question_data (df_Question_schoolID) having colun School_ID
        df_Student_Submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_item')
        df_Student_Submissions_schoolID = df_Student_Submissions.select("Item_ID","School_ID")
        df_Question_schoolID = df_Question.join(df_Student_Submissions_schoolID,on=["Item_ID"],how="left")
        df_Question_schoolID = df_Question_schoolID.select(
            [df_Question[col] for col in df_Question.columns] + [df_Student_Submissions_schoolID["School_ID"]])
       
       # dim_question_data     
        df_Question_schoolID = df_Question_schoolID.drop("Standards")
        df_Question_schoolID = df_Question_schoolID.withColumnRenamed('Standards_Val','Standard')

        # df_dim_question_data = df_Question_schoolID[['Question',
        # 'Position_Number',
        # 'Item_ID',
        # 'Item_Name',
        # 'School_ID',
        # 'Standard',
        # 'Question_ID',
        # 'Question_No',        
        # 'Least_Points_Earned',
        # 'Correct_Answer',
        # 'Question_Type',
        # 'Average_Points_Earned',
        # 'Associated_Question_ID',
        # 'Total_Points',
        # 'Most_Points_Earned',
        # 'Correctly_Answered',
        # 'Sub-Question',
        # 'Session',
        # 'Assessment_type',
        # 'Subject',
        # 'Grade',
        # # 'Section_NID',
        # 'Section'
        # ]].drop_duplicates()
        # df_dim_question_data = df_dim_question_data.withColumn("Grade", F.when((F.col("School_ID") == "554425139")& (F.col("Grade").isin(["Grade 9", "Grade 10", "Grade 11", "Grade 12"])),"Regular 9–12").otherwise(F.col("Grade")))
        # standard = oea.load(f'stage3/Published/schoology/v{self.version}/dim_standard')
        # standard = standard.withColumnRenamed("Schoology_Standard", "Standards")
        # dim_standard = standard[['Identifier', 'Standards']]
        # df_dim_question_data = df_dim_question_data.join(
        #     dim_standard,
        #     df_dim_question_data.Standard.contains(dim_standard.Standards),
        #     how="left"
        # )
        # # df_dim_question_data = df_dim_question_data.withColumn('Qkey',concat(df_dim_question_data['Session'],df_dim_question_data['Assessment_type'],df_dim_question_data['Subject'],df_dim_question_data['Grade'],df_dim_question_data['Question_ID']))
        # df_dim_question_data = df_dim_question_data.withColumn(
        #     'Qkey',
        #     F.concat(
        #         F.when(F.col('Session').isNotNull(), F.col('Session')).otherwise('DEFAULT_SESSION'),
        #         F.when(F.col('Assessment_type').isNotNull(), F.col('Assessment_type')).otherwise('DEFAULT_ASSESSMENT'),
        #         F.when(F.col('Subject').isNotNull(), F.col('Subject')).otherwise('DEFAULT_SUBJECT'),
        #         F.when(F.col('Grade').isNotNull(), F.col('Grade')).otherwise('DEFAULT_GRADE'),
        #         F.when(F.col('Question_ID').isNotNull(), F.col('Question_ID')).otherwise('DEFAULT_QID'),
        #         F.when(F.col("Position_Number").isNotNull(), F.col("Position_Number")).otherwise(F.lit("DEFAULT_POSITION_NO")),
        #         F.when(F.col('Correct_Answer').isNotNull(), F.col('Correct_Answer')).otherwise('DEFAULT_CORRECT_ANSWER'),
        #         F.when(F.col('Standard').isNotNull(), F.col('Standard')).otherwise('DEFAULT_STANDARD'),
        #         F.when(F.col("School_ID").isNotNull(), F.col("School_ID")).otherwise(F.lit("DEFAULT_SCHOOL_ID"))
        #     )
        # )
        # df_dim_question_data = df_dim_question_data.drop_duplicates()
        # df_dim_question_data = df_dim_question_data.withColumn('Ukey',generate_uuid_8(df_dim_question_data['School_ID'],df_dim_question_data['Session'],df_dim_question_data['Assessment_type'],df_dim_question_data['Item_Name'],df_dim_question_data['Subject'],df_dim_question_data['Grade'],df_dim_question_data['Question'],df_dim_question_data['Correct_Answer']))
       
        # # Replace 'Â' with space in the 'Question' column
        # # df_dim_question_data = df_dim_question_data.withColumn(
        # #     'Question',
        # #     regexp_replace('Question', 'Â', ' ')
        # # )

        # # # Replace 'Â' with space in the 'Correct_Answer' column
        # # df_dim_question_data = df_dim_question_data.withColumn(
        # #     'Correct_Answer',
        # #     regexp_replace('Correct_Answer', 'Â', ' ')
        # # )
        # # params = {'req_cols': ['School_ID','Question_ID', 'Standards']}
        # params = {
        #     'req_cols': [
        #         ['School_ID', 'Question_ID', 'Standard'],
        #         ['School_ID', 'Question_ID', 'Correct_Answer']
        #     ]
        # }
        # self.delete_stale_rows(df_dim_question_data,f'stage2/Enriched/schoology/v{self.version}/dim_question_data',f'stage3/Published/schoology/v{self.version}/dim_question_data',params)
        # self.publish(df_dim_question_data,f'stage2/Enriched/schoology/v{self.version}/dim_question_data',f'stage3/Published/schoology/v{self.version}/dim_question_data', primary_key='Qkey')
        # oea.add_to_lake_db(f'stage2/Enriched/schoology/v{self.version}/dim_question_data')
        # oea.add_to_lake_db(f'stage3/Published/schoology/v{self.version}/dim_question_data')

        df_dim_question_data = df_Question_schoolID[['Question',
        'Position_Number',
        'Item_ID',
        'Item_Name',
        'School_ID',
        'Standard',
        'Question_ID',
        'Question_No',        
        'Least_Points_Earned',
        'Correct_Answer',
        'Question_Type',
        'Average_Points_Earned',
        'Associated_Question_ID',
        'Total_Points',
        'Most_Points_Earned',
        'Correctly_Answered',
        'Sub-Question',
        'Session',
        'Assessment_type',
        'Subject',
        'Grade',
        # 'Section_NID',
        'Section'
        ]].drop_duplicates()
        # df_dim_question_data = df_dim_question_data.withColumn("Grade", F.when((F.col("School_ID") == "554425139")& (F.col("Grade").isin(["Grade 9", "Grade 10", "Grade 11", "Grade 12"])),"Regular 9–12").otherwise(F.col("Grade")))
        standard = oea.load(f'stage2/Ingested/canvas/v{schoology.version}/standards')
        standard = standard.withColumnRenamed("Schoology_Standard", "Standards")
        dim_standard = standard[['Identifier', 'Standards']]
        df_dim_question_data = df_dim_question_data.join(
            dim_standard,
            df_dim_question_data.Standard.contains(dim_standard.Standards),
            how="left"
        )
        # df_dim_question_data = df_dim_question_data.withColumn('Qkey',concat(df_dim_question_data['Session'],df_dim_question_data['Assessment_type'],df_dim_question_data['Subject'],df_dim_question_data['Grade'],df_dim_question_data['Question_ID']))
        df_dim_question_data = df_dim_question_data.withColumn(
            'Qkey',
            F.concat(
                F.when(F.col('Session').isNotNull(), F.col('Session')).otherwise('DEFAULT_SESSION'),
                F.when(F.col('Assessment_type').isNotNull(), F.col('Assessment_type')).otherwise('DEFAULT_ASSESSMENT'),
                F.when(F.col('Subject').isNotNull(), F.col('Subject')).otherwise('DEFAULT_SUBJECT'),
                F.when(F.col('Grade').isNotNull(), F.col('Grade')).otherwise('DEFAULT_GRADE'),
                F.when(F.col('Question_ID').isNotNull(), F.col('Question_ID')).otherwise('DEFAULT_QID'),
                F.when(F.col("Position_Number").isNotNull(), F.col("Position_Number")).otherwise(F.lit("DEFAULT_POSITION_NO")),
                F.when(F.col('Correct_Answer').isNotNull(), F.col('Correct_Answer')).otherwise('DEFAULT_CORRECT_ANSWER'),
                F.when(F.col('Standard').isNotNull(), F.col('Standard')).otherwise('DEFAULT_STANDARD'),
                F.when(F.col("School_ID").isNotNull(), F.col("School_ID")).otherwise(F.lit("DEFAULT_SCHOOL_ID"))
            )
        )
        df_dim_question_data = df_dim_question_data.drop_duplicates()
        df_dim_question_data = df_dim_question_data.withColumn('Ukey',generate_uuid_8(df_dim_question_data['School_ID'],df_dim_question_data['Session'],df_dim_question_data['Assessment_type'],df_dim_question_data['Item_Name'],df_dim_question_data['Subject'],df_dim_question_data['Grade'],df_dim_question_data['Question'],df_dim_question_data['Correct_Answer']))
       
        # Replace 'Â' with space in the 'Question' column
        # df_dim_question_data = df_dim_question_data.withColumn(
        #     'Question',
        #     regexp_replace('Question', 'Â', ' ')
        # )

        # # Replace 'Â' with space in the 'Correct_Answer' column
        # df_dim_question_data = df_dim_question_data.withColumn(
        #     'Correct_Answer',
        #     regexp_replace('Correct_Answer', 'Â', ' ')
        # )
        # params = {'req_cols': ['School_ID','Question_ID', 'Standards']}
        params = {
            'req_cols': [
                ['School_ID', 'Question_ID', 'Standard'],
                ['School_ID', 'Question_ID', 'Correct_Answer']
            ]
        }
        self.delete_stale_rows(df_dim_question_data,f'stage2/Enriched/canvas/v{self.version}/dim_question_data1',f'stage3/Published/canvas/v{self.version}/dim_question_data1',params)
        self.publish(df_dim_question_data,f'stage2/Enriched/canvas/v{self.version}/dim_question_data1',f'stage3/Published/canvas/v{self.version}/dim_question_data1', primary_key='Qkey')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_question_data1')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_question_data1')
        
        # # dim_standard
        df_dim_standard = oea.load(f'stage2/Ingested/canvas/v{schoology.version}/standards')
        # self.publish(df_dim_standard, f'stage2/Enriched/canvas/v{self.version}/dim_standard',f'stage3/Published/canvas/v{self.version}/dim_standard', primary_key='Identifier')
        # oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_standard')
        # oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_standard')
        question_data = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_question_data1')
        question_data = question_data.groupBy("Standard").agg(count("*").alias("count"))
        question_data = question_data.drop("Identifier","Subject")
        standard = df_dim_standard
        q = question_data.alias("q")
        s = standard.alias("s")

        # Perform partial string match join
        # df_joined = q.join(
        #     s,
        #     expr("s.Schoology_Standard LIKE concat(q.Standard, '%')"),
        #     "left"
        # )
        df_dim_question_data1 = question_data.join(
                standard,
                question_data.Standard.contains(standard.Schoology_Standard),
                how="left"
            )
        df_dim_question_data1 = df_dim_question_data1["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"].dropDuplicates(["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"])
        df_dim_question_data = standard.join(
                question_data,
                standard.Schoology_Standard.contains(question_data.Standard),
                how="left"
            )
        df_dim_question_data = df_dim_question_data["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"].dropDuplicates(["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"])    
        df_dim_question_data = df_dim_question_data.union(df_dim_question_data1)
        # df_dim_question_data.show()    
        # Show the result
        dim_strand = df_dim_question_data["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"].dropDuplicates(["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"])
        # dim_strand = dim_strand.filter(dim_strand.Standard == "SCI.5.SC.5.P.13.1")
        # dim_strand.show()
        dim_strand = df_dim_question_data["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"].dropDuplicates(["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"])

        dim_strand = dim_strand.withColumnRenamed('Standard','Schoology_Standard')
        dim_strand = dim_strand.union(df_dim_standard["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Schoology_Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"]).na.drop()
        dim_strand = dim_strand["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Schoology_Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"].dropDuplicates(["Cognitive_Complexity_Rating", "Direct_Link","Grader","Identifier","Language","Schoology_Standard","Standard_New","Strand","Subject","cluster","description","lastChangeDateTime","rundate"])
        dim_strand = dim_strand.filter(~col("Schoology_Standard").rlike(r"^\d+$"))

        dim_strand = dim_strand.dropDuplicates(["Schoology_Standard"])
        dim_strand = dim_strand.withColumn(
            "cPalms_Standard",
            F.expr("concat_ws('.', slice(split(Schoology_Standard, '\\\\.'), 3, size(split(Schoology_Standard, '\\\\.' ))))")
        )
        dim_strand = dim_strand.withColumn(
            "uniquesID",
            concat_ws("_", dim_strand.Identifier, dim_strand.Schoology_Standard)
        )
        dim_strand = dim_strand.withColumn(
            "uniquesID",
            when(
                (col("Identifier") == "Other") | (col("Schoology_Standard") == "Other"),
                "Other"
            ).otherwise(
                concat_ws("_", col("Identifier"), col("Schoology_Standard"))
            )
        )
        schoology.publish(dim_strand, f'stage2/Enriched/canvas/v{schoology.version}/dim_standard',f'stage3/Published/canvas/v{schoology.version}/dim_standard', primary_key='uniquesID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/dim_standard')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/dim_standard')

        # dim_strand
        question_data = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_question_data1')
        question_data = question_data.drop("Identifier")
        standard = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_standard')
        q = question_data.alias("q")
        s = standard.alias("s")

        # Perform partial string match join
        # df_joined = q.join(
        #     s,
        #     expr("s.Schoology_Standard LIKE concat(q.Standard, '%')"),
        #     "left"
        # )
        df_dim_question_data = standard.join(
                question_data,
                standard.Schoology_Standard.contains(question_data.Standard),
                how="left"
            )
        # df_dim_question_data.show()
        # Show the result
        dim_strand = df_dim_question_data["Identifier", "Strand"].dropDuplicates(["Identifier", "Strand"])
        # dim_strand = dim_strand.withColumn("strand_ID", generate_uuid_2(dim_strand['Identifier'], dim_strand['Strand']))
        dim_strand = dim_strand.withColumn("ID", monotonically_increasing_id())
        dim_strand = dim_strand.withColumn("strand_ID", generate_uuid_2(dim_strand['Identifier'], dim_strand['Strand']))
        df_filtered = dim_strand.filter(col("Standard") == "MA.7.DP.2")
        df_filtered.show()
        schoology.publish(dim_strand, f'stage2/Enriched/canvas/v{schoology.version}/dim_strand',f'stage3/Published/canvas/v{schoology.version}/dim_strand', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/dim_strand')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/dim_strand')


        

        # dim_unit_lesson
        dim_unit_lesson=df_Student_Submissions[['Item_ID','Item_Name','School_ID',]].drop_duplicates()
        self.publish(dim_unit_lesson, f'stage2/Enriched/canvas/v{self.version}/dim_unit_lesson',f'stage3/Published/canvas/v{self.version}/dim_unit_lesson', primary_key='Item_ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_unit_lesson')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_unit_lesson')



        # # dim_section
        df_dim_section = fact_Student_Submissions[['Section_Code','Item_ID','Section_NID','Section_Name','Section_Instructors','School_ID']].drop_duplicates().drop_duplicates().dropna(subset=['Section_Code','Item_ID','Section_NID','Section_Name','Section_Instructors','School_ID'])
        self.publish(df_dim_section,f'stage2/Enriched/canvas/v{self.version}/dim_section',f'stage3/Published/canvas/v{self.version}/dim_section', primary_key='Section_NID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_section')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_section')

        dim_session = fact_Student_Submissions[['School_ID','Session']].drop_duplicates()


        dim_session = dim_session.withColumn("session_ID", generate_uuid_2(dim_session['School_ID'], dim_session['Session']))
        dim_session = dim_session.dropna(subset=["Session_ID", "School_ID", "Session"])

        self.publish(dim_session, f'stage2/Enriched/canvas/v{self.version}/dim_session',f'stage3/Published/canvas/v{self.version}/dim_session', primary_key='Session_ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_session')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_session')


       

        # dim_grade
        dim_grade = fact_Student_Submissions[['Grade_ID','School_ID','Grade']].drop_duplicates().dropna(subset=['Grade_ID','School_ID','Grade'])
        self.publish(dim_grade,f'stage2/Enriched/canvas/v{self.version}/dim_grade',f'stage3/Published/canvas/v{self.version}/dim_grade', primary_key='Grade_ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_grade')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_grade')

        # # dim_assessment_type
        dim_assessment_type = fact_Student_Submissions[['Assessment_ID','School_ID','Assessment_type']].drop_duplicates().dropna(subset=['Assessment_ID','School_ID','Assessment_type'])
        self.publish(dim_assessment_type,f'stage2/Enriched/canvas/v{self.version}/dim_assessment_type',f'stage3/Published/canvas/v{self.version}/dim_assessment_type', primary_key='Assessment_ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_assessment_type')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_assessment_type')

        # # dim_subject
        dim_subject = fact_Student_Submissions[['Subject_ID','School_ID','Subject','Assessment_type','Grade',"Session","Item_Name"]].drop_duplicates().dropna(subset=['Subject_ID','School_ID','Subject','Assessment_type','Grade',"Session","Item_Name"])
        self.publish(dim_subject,f'stage2/Enriched/canvas/v{self.version}/dim_subject',f'stage3/Published/canvas/v{self.version}/dim_subject', primary_key='Subject_ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/dim_subject')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/dim_subject')

    def build_fact_tables(self,tables_source):
        """ Stage2/Ingested/canvas/{version}/{item}  ==> Stage2/Enriched/canvas/{version}/{item}
        """
                        
        df_Student_Submissions = oea.load(f'stage2/Ingested/canvas/v{schoology.version}/student_submissions')
        df_Student_Submissions = df_Student_Submissions.withColumn("rundate", F.to_date("rundate", "yyyy-MM-dd"))
        key_columns = [c for c in df_Student_Submissions.columns 
                if c not in ("rundate", "Unique_Key","Points_Received","Submission_Grade","File_Name")]

        w = Window.partitionBy(key_columns).orderBy(F.col("rundate").desc())

        df_Student_Submissions = (
                df_Student_Submissions
                    .withColumn("rn", F.row_number().over(w))
                    .filter("rn = 1")
                    .drop("rn")
        )


        df_Student_Submissions = df_Student_Submissions.withColumnRenamed('User_School_ID','School_ID').withColumnRenamed('User_School_Name','School_Name')

        df_Question = oea.load(f'stage2/Ingested/canvas/v{self.version}/question_data')
        df_Question = df_Question.filter((F.col('Standards_Val').isNull()) & (F.col('Standards') != 'Standards17') | (F.col('Standards').isNull() & F.col('Standards_Val').isNull()))
        def generate_uuid_2(column1, column2):
            combined_column = concat_ws("_",column1, column2)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid

        def generate_uuid_5(column1, column2,column3, column4, column5):
            combined_column = concat_ws("_",column1,column2,column3,column4, column5)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid

        def generate_uuid_6(column1, column2,column3 ,column4, column5, column6):
            combined_column = concat_ws("_",column1,column2,column3,column4,column5,column6)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid
        def generate_uuid_7(column1, column2,column3, column4,column5,column6,column7):
            combined_column = concat_ws("_",column1, column2,column3, column4,column5,column6,column7)
            uuid = sha2(combined_column, 256)  # Apply a hashing function to generate UUIDs
            return uuid  

        # fact_student_submission          
        df_Student_Submissions = df_Student_Submissions[['User_UID','First_Name','Last_Name','User_Role_ID','School_ID','Course_NID','Section_NID',
        'Section_Code','Item_ID','Item_Name','First_Access','Latest_Attempt','Total_Time','Submission_Grade','Submission','Question_ID',
        'Session','Assessment_type','Subject','Grade','Section','File_Name',
        'Position_Number','Sub-Question','Answer_Submission','Correct_Answer','Points_Received','Points_Possible']]
        #adding one new column to be a primary key
        df_Student_Submissions = df_Student_Submissions.withColumn("User_id_ques_id",concat(col("User_UID"), lit('-'), col("Question_ID")))
        df_Student_Submissions = df_Student_Submissions.withColumn("User_Name",concat(col("First_Name"), lit(' '), col("Last_Name")))
        df_Student_Submissions = df_Student_Submissions.withColumn('Qkey', 
            concat(col('Session'), col('Assessment_type'), col('Subject'), col('Grade'), col('Question_ID'))
        )    

        # Selecting required columns 
        fact_Student_Submissions = df_Student_Submissions.select(
            'User_UID','User_Name','User_Role_ID','School_ID','Course_NID','Section_NID',
        'Section_Code','Item_ID','Item_Name','First_Access','Latest_Attempt','Total_Time','Submission_Grade','Submission','Question_ID',
        'Session','Assessment_type','Subject','Grade','Section','File_Name',
        'Position_Number','Sub-Question','Answer_Submission','Correct_Answer','Points_Received','Points_Possible','User_id_ques_id'
        )

        # 3 dim tables (dim_grade,dim_assessment_type, dim_subject) after student submissions fact table
        fact_Student_Submissions = fact_Student_Submissions.withColumn("Grade_ID", generate_uuid_2(df_Student_Submissions['School_ID'], df_Student_Submissions['Grade']))
        fact_Student_Submissions = fact_Student_Submissions.withColumn("Assessment_ID", generate_uuid_2(df_Student_Submissions['School_ID'], df_Student_Submissions['Assessment_type']))
        fact_Student_Submissions = fact_Student_Submissions.withColumn("Subject_ID", generate_uuid_6(df_Student_Submissions['School_ID'], df_Student_Submissions['Subject'], df_Student_Submissions['Assessment_type'], df_Student_Submissions['Grade'], df_Student_Submissions['Session'],df_Student_Submissions['Item_Name']))
        questiondata = oea.load(f'stage3/Published/canvas/v{self.version}/dim_question_data1')  
        standard = oea.load(f'stage3/Published/canvas/v{self.version}/dim_standard')
        strand = oea.load(f'stage3/Published/canvas/v{self.version}/dim_strand')
        dim_strand = strand[['Identifier', 'strand_ID']]
        standard = standard.withColumnRenamed("Schoology_Standard", "Standard")
        dim_standard = standard[['Identifier', 'Standard']]
        ques_stand = questiondata[["Question_ID","Standard"]].dropDuplicates()
        fact_Student_Submissions = fact_Student_Submissions.join(ques_stand,on=["Question_ID"],how="left")    
        fact_Student_Submissions = fact_Student_Submissions.join(
            dim_standard,
            on=["Standard"],
            how="left"
        )
        
        fact_Student_Submissions = fact_Student_Submissions.join(
            dim_strand,
            on=["Identifier"],
            how="left"
        )
        fact_Student_Submissions = fact_Student_Submissions.withColumn(
            "User_id_ques_id_stand",
            concat(
                when(col("School_ID").isNotNull(), col("School_ID")).otherwise(lit("DEFAULT_SCHOOLID")),
                lit('-'),
                when(col("User_UID").isNotNull(), col("User_UID")).otherwise(lit("DEFAULT_USER")),
                lit('-'),
                when(col("Question_ID").isNotNull(), col("Question_ID")).otherwise(lit("DEFAULT_QID")),
                lit('-'),
                when(col("Position_Number").isNotNull(), col("Position_Number")).otherwise(lit("DEFAULT_POS")),
                lit('-'),
                when(col("Answer_Submission").isNotNull(), col("Answer_Submission")).otherwise(lit("DEFAULT_ANSWER_SUBMISSION")),
                lit('-'),
                when(col("Points_Possible").isNotNull(), col("Points_Possible")).otherwise(lit("DEFAULT_POINTS_REC")),
                lit('-'),
                when(col("Submission").isNotNull(), col("Submission")).otherwise(lit("Submission")),
                lit('-'),
                when(col("Standard").isNotNull(), col("Standard")).otherwise(lit("DEFAULT_STANDARD"))
            )
        )
        # fact_Student_Submissions.filter(fact_Student_Submissions["Item_ID"]=='7338033712').select("Standard", "Question_ID","User_id_ques_id").distinct().show(100,truncate=False)
        params = {
        'req_cols': [
            ['School_ID', 'Question_ID', 'Standard']
        ]
    }
        self.delete_stale_rows(fact_Student_Submissions,f'stage2/Enriched/canvas/v{self.version}/fact_student_submission',f'stage3/Published/canvas/v{self.version}/fact_student_submission',params)
        self.publish(fact_Student_Submissions,f'stage2/Enriched/canvas/v{self.version}/fact_student_submission',f'stage3/Published/canvas/v{self.version}/fact_student_submission', primary_key='User_id_ques_id_stand')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/fact_student_submission')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/fact_student_submission')
    def build_pseudomyzed_tables(self):
        from pyspark.sql import functions as F
        # dim_section_Hash
        fact_student_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/fact_student_submission')
        DimItemName = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_item')
        fact_student_submissions=fact_student_submissions.join(DimItemName,on=['Item_ID'],how='left')
        fact_student_submissions_Unique = fact_student_submissions.groupby("Section_NID","Section_Instructors").agg(sum("Item_ID"))
        fact_student_submissions_Unique = fact_student_submissions_Unique.select("Section_NID","Section_Instructors").withColumn("TeacherName_Hash", concat(F.lit("Teacher_Name "), monotonically_increasing_id()))
        schoology.publish(fact_student_submissions_Unique, f'stage2/Enriched/canvas/v{schoology.version}/dim_section_Hash',f'stage3/Published/canvas/v{schoology.version}/dim_section_Hash', primary_key='Section_NID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/dim_section_Hash')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/dim_section_Hash')

        fact_student_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/fact_student_submission')
        fact_student_submissions_Unique = fact_student_submissions.groupby("User_UID","User_Name").agg(sum("Item_ID"))
        fact_student_submissions_Unique = fact_student_submissions_Unique.select("User_UID","User_Name").withColumn("StudentName_Hash", concat(F.lit("Student_Name "), monotonically_increasing_id()))
        schoology.publish(fact_student_submissions_Unique, f'stage2/Enriched/canvas/v{schoology.version}/dim_student_hash',f'stage3/Published/canvas/v{schoology.version}/dim_student_hash', primary_key='User_UID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/dim_student_hash')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/dim_student_hash')

        fact_student_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/fact_student_submission')
        fact_student_submissions_Unique =oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_student_hash')
        fact_student_submissions_Unique= fact_student_submissions_Unique.select(F.col("User_UID"),F.col("StudentName_Hash"))
        fact_student_submissions=fact_student_submissions.join(fact_student_submissions_Unique,on=["User_UID"],how="left")
        schoology.publish(fact_student_submissions, f'stage2/Enriched/canvas/v{schoology.version}/fact_student_submissions_Hash',f'stage3/Published/canvas/v{schoology.version}/fact_student_submissions_Hash', primary_key='User_id_ques_id_stand')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/fact_student_submissions_Hash')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/fact_student_submissions_Hash')

    def cube_Build(self):
        from pyspark.sql import functions as F
        from pyspark.sql.functions import col, avg, sum
        self.build_pseudomyzed_tables()
        Fact_student_submissions = oea.load(f'stage3/Published/canvas/v{self.version}/fact_student_submission')
        Fact_student_submissions = Fact_student_submissions.withColumn(
            "Total_Seconds",
            F.expr(
                "int(split(Total_Time, ':')[0])*3600 + "
                "int(split(Total_Time, ':')[1])*60 + "
                "int(split(Total_Time, ':')[2])"
            )
        )
        Fact_student_submissions = Fact_student_submissions.withColumnRenamed('Standard', 'Standards')

        Cube_User_Grade = Fact_student_submissions.groupby('School_ID','Subject_ID','User_UID','Item_ID').agg(
            (sum("Points_Received") / sum("Points_Possible")).alias("Grade_Average")
        )
        Cube_Grade_measure = Fact_student_submissions.groupby('School_ID','Subject_ID','User_UID','Item_ID').agg(
                (sum("Points_Received") / sum("Points_Possible")).alias("Grade_Average"),
        )
        Cube_Grade_Summary = Cube_Grade_measure.rollup('School_ID','Subject_ID','Item_ID').agg(
                        (avg("Grade_Average")).alias("Grade_Average"),
                        ((1 - (avg(F.col('Grade_Average'))))).alias('Percentage_InCorrect_Answers'),
                        (F.min("Grade_Average")).alias("Grade_Min"),
                        (F.max("Grade_Average")).alias("Grade_Max")
        )
        # Cube_Grade_Summary = Cube_Grade_Summary.withColumn("ID", monotonically_increasing_id())
        Cube_Grade_Summary = Cube_Grade_Summary.withColumn(
            "ID",
            F.sha2(
                F.concat(
                    F.when(F.col("School_ID").isNotNull(), F.col("School_ID")).otherwise(F.lit("DEFAULT_SCHOOL_ID")),
                    F.when(F.col("Item_ID").isNotNull(), F.col("Item_ID")).otherwise(F.lit("DEFAULT_ITEM_ID"))
                ), 
                256
            )
        )

        self.publish(Cube_Grade_Summary, f'stage2/Enriched/canvas/v{self.version}/Cube_Grade_Summary',f'stage3/Published/canvas/v{self.version}/Cube_Grade_Summary', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/Cube_Grade_Summary')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/Cube_Grade_Summary')
        from pyspark.sql.functions import col, sum, countDistinct, sha2, concat

        Cube_School_Summary = Fact_student_submissions.rollup('School_ID','Subject_ID','Item_ID').agg(
            countDistinct("Question_ID").alias("Total_Questions"),
            countDistinct("Identifier").alias("Total_Standards"),
            countDistinct("User_UID").alias("Total_Students"),
            sum("Points_Possible").alias("Total_Possible_Point"),
            sum('Points_Received').alias("Total_Score"),
            (sum("Points_Received") / sum("Points_Possible")).alias("Grade_Average"),
            ((1 - (sum(F.col('Points_Received')) / sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers')
        )
        # Cube_School_Summary = Cube_School_Summary.withColumn("ID", monotonically_increasing_id())
        Cube_School_Summary = Cube_School_Summary.withColumn(
            "ID",
            F.sha2(
                F.concat(
                    F.when(F.col("School_ID").isNotNull(), F.col("School_ID")).otherwise(F.lit("DEFAULT_SCHOOL_ID")),
                    F.when(F.col("Item_ID").isNotNull(), F.col("Item_ID")).otherwise(F.lit("DEFAULT_ITEM_ID"))
                ), 
                256
            )
        )

        self.publish(Cube_School_Summary, f'stage2/Enriched/canvas/v{self.version}/Cube_School_Summary',f'stage3/Published/canvas/v{self.version}/Cube_School_Summary', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/Cube_School_Summary')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/Cube_School_Summary')
        Cube_Standard_Summary = Fact_student_submissions.rollup('Item_ID','Strand_ID','Identifier').agg(
            countDistinct("Question_ID").alias("Total_Questions"),
            countDistinct("Identifier").alias("Total_Standards"),
            sum("Points_Possible").alias("Total_Possible_Point"),
            sum('Points_Received').alias("Total_Score"),
            (sum("Points_Received") / sum("Points_Possible")).alias("Grade_Average"),
            ((1 - (sum(F.col('Points_Received')) / sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers'),
        )
        # Cube_Standard_Summary = Cube_Standard_Summary.withColumn("ID", monotonically_increasing_id())
        Cube_Standard_Summary = Cube_Standard_Summary.withColumn(
            "ID",
            F.sha2(
                concat(
                    when(col("Item_ID").isNotNull(), col("Item_ID")).otherwise(lit("DEFAULT_ITEM_ID")),
                    when(col("Strand_ID").isNotNull(), col("Strand_ID")).otherwise(lit("DEFAULT_STRAND_ID")),
                    when(col("Identifier").isNotNull(), col("Identifier")).otherwise(lit("DEFAULT_IDENTIFIER"))
                ),
                256
            )
        )
        self.publish(Cube_Standard_Summary, f'stage2/Enriched/canvas/v{self.version}/Cube_Standard_Summary',f'stage3/Published/canvas/v{self.version}/Cube_Standard_Summary', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/Cube_Standard_Summary')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/Cube_Standard_Summary')
        
        # Cube Question Summary
        # Cube_Question_Summary = Fact_student_submissions.groupBy('Item_ID', 'Question_ID').agg(
        #     F.sum("Points_Possible").alias("Total_Possible_Point"),
        #     F.sum('Points_Received').alias("Total_Score"),
        #     (F.sum("Points_Received") / F.sum("Points_Possible")).alias("Grade_Average"),
        #     ((1 - (F.sum(F.col('Points_Received')) / F.sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers')
        # )
        # # Filter out submissions for each choice and include Points_Received and Student_Name
        # ChoiceSubmissions = Fact_student_submissions.select("Question_ID", "Answer_Submission", "Points_Received", "User_Name")
        # # Total submissions per question
        # TotalSubmissions = ChoiceSubmissions.groupBy("Question_ID").agg(F.count("Answer_Submission").alias("Total_Submissions"))
        # # Calculate the percentage of each choice being selected
        # ChoicePercentage = ChoiceSubmissions.groupBy("Question_ID", "Answer_Submission").agg(
        #     F.count("Answer_Submission").alias("Choice_Submission_Count"),
        #     F.min("Points_Received").alias("Points_Received")  # Keep Points_Received for correct/incorrect marking
        # ).join(TotalSubmissions, on="Question_ID").withColumn(
        #     "Choice_Percentage", F.col("Choice_Submission_Count") / F.col("Total_Submissions")
        # )
        # # Mark if a choice is correct or incorrect based on Points_Received
        # ChoiceWithCorrectness = ChoicePercentage.withColumn(
        #     "Is_Correct", F.when(F.col("Points_Received") > 0, F.lit("Correct")).otherwise(F.lit("Incorrect"))
        # )
        # # Filter only incorrect choices
        # IncorrectChoices = ChoiceWithCorrectness.filter(F.col("Is_Correct") == "Incorrect")
        # # Create a list of student names for each incorrect choice
        # StudentChoices = ChoiceSubmissions.filter(F.col("Points_Received") == 0).groupBy("Question_ID", "Answer_Submission").agg(
        #     F.collect_list("User_Name").alias("Students_Who_Chose")
        # )
        # # Join the student names back with the incorrect choices
        # IncorrectChoicesWithStudents = IncorrectChoices.join(StudentChoices, on=["Question_ID", "Answer_Submission"])
        # # Concatenate results to show percentages for only incorrect choices and student names
        # FinalResults = IncorrectChoicesWithStudents.withColumn(
        #     "Choice_Details", F.expr("""
        #         CASE
        #             WHEN size(Students_Who_Chose) = 1 THEN
        #                 CONCAT(
        #                     FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
        #                 )
        #             ELSE
        #                 CONCAT(
        #                     FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
        #                 )
        #         END
        #     """)
        # )
        # FinalResults = FinalResults.withColumn(
        #     "Choice_Details_WithName", F.expr("""
        #         CASE
        #             WHEN size(Students_Who_Chose) = 1 THEN
        #                 CONCAT(
        #                     '  [', Answer_Submission, '] (',
        #                     Students_Who_Chose[0], ')'
        #                 )
        #             ELSE
        #                 CONCAT(
        #                     '  [', Answer_Submission, '] (',
        #                     CONCAT_WS(', ', Students_Who_Chose), ')'
        #                 )
        #         END
        #     """)
        # )
        # # Aggregating incorrect choices into a single string per question
        # ChoicesPerQuestion = FinalResults.groupBy(
        #     'Question_ID'
        # ).agg(
        #     F.concat_ws(", ", F.collect_list("Choice_Details")).alias("Incorrect_Choice_Details"),
        #     F.concat_ws(", ", F.collect_list("Choice_Details_WithName")).alias("Incorrect_Details_Name")
        # ).withColumn(
        #     "Incorrect_Choice_Details", F.expr("substring(Incorrect_Choice_Details, 1, 8000)")
        # ).withColumn(
        #     "Incorrect_Details_Name", F.expr("substring(Incorrect_Details_Name, 1, 8000)")
        # )
        # # Join with Cube_Question_Summary to include incorrect choice details
        
        # DimQuestionData = oea.load(f'stage3/Published/schoology/v{self.version}/dim_question_data')
        # DimItemName = oea.load(f'stage3/Published/schoology/v{self.version}/dim_item')

        # # Join Fact_student_submissions with DimQuestionData on Question_ID and Item_ID
        # JoinedResults = Cube_Question_Summary.join(ChoicesPerQuestion, on="Question_ID", how="left")
        # JoinedResults = JoinedResults.join(
        #     DimQuestionData,
        #     on=["Question_ID", "Item_ID"],
        #     how="left"  # Adjust join type as needed (inner, left, etc.)
        # )
        # # JoinedResults.show()
        # JoinedResults = JoinedResults.join(
        #     DimItemName,
        #     on=['School_ID', "Item_ID","Item_Name"],
        #     how="left"  # Adjust join type as needed (inner, left, etc.)
        # )
        # JoinedResults = JoinedResults.withColumn(
        #     "Question_no_url",
        #     trim(regexp_replace(F.col("Question"), r"<[^>]*>", ""))
        # )
        # JoinedResults = JoinedResults.withColumn(
        #     "Standard", 
        #     F.when(F.col("Standard").isNull() | (F.col("Standard") == "null"), "Other")
        #     .otherwise(F.col("Standard"))
        # )
        # JoinedResults = JoinedResults.withColumn(
        #     "Standards", 
        #     F.when(F.col("Standards").isNull() | (F.col("Standards") == "null"), "Other")
        #     .otherwise(F.col("Standards"))
        # )
        # all_columns = JoinedResults.columns
        # group_by_columns = ['Subject_ID','Question_ID', 'Question_No', 'Grade_Average']
        # agg_expr = []
        
        # for col in all_columns:
        #     if col in group_by_columns:
        #         continue  # Skip the columns that we are grouping by
        #     elif col == 'Standard' or col == 'Correct_Answer':
        #         agg_expr.append(F.concat_ws(',', F.collect_set(col)).alias(col))
        #     else:
        #         agg_expr.append(F.first(col).alias(col))
        # JoinedResults = JoinedResults.groupBy(*group_by_columns).agg(*agg_expr)
        
        # # Cube_Question_Summary = JoinedResults.withColumn("ID", monotonically_increasing_id())
        # Cube_Question_Summary = JoinedResults.withColumn(
        #     "ID",
        #     F.sha2(
        #         F.concat(
        #             F.when(F.col("School_ID").isNotNull(), F.col("School_ID")).otherwise(F.lit("DEFAULT_SCHOOL_ID")),
        #             F.when(F.col("Item_ID").isNotNull(), F.col("Item_ID")).otherwise(F.lit("DEFAULT_ITEM_ID")),
        #             F.when(F.col("Item_Name").isNotNull(), F.col("Item_Name")).otherwise(F.lit("DEFAULT_ITEM_NAME")),
        #             F.when(F.col("Question_ID").isNotNull(), F.col("Question_ID")).otherwise(F.lit("DEFAULT_QUESTION_ID"))
        #             # ,
        #             # F.when(F.col("Standard").isNotNull(), F.col("Standard")).otherwise(F.lit("DEFAULT_STANDARD")),
        #             # F.when(F.col("Correct_Answer").isNotNull(), F.col("Correct_Answer")).otherwise(F.lit("DEFAULT_CORRECT_ANSWER"))
        #         ), 
        #         256
        #     )
        # )

        # params = {
        #     'req_cols': [
        #         ['School_ID', 'Question_ID', 'Standard'],
        #         ['School_ID', 'Question_ID', 'Correct_Answer']
        #     ]
        # }    
        # self.delete_stale_rows(Cube_Question_Summary, f'stage2/Enriched/schoology/v{self.version}/Cube_Question_Summary_',f'stage3/Published/schoology/v{self.version}/Cube_Question_Summary_', params)
        # self.publish(Cube_Question_Summary, f'stage2/Enriched/schoology/v{self.version}/Cube_Question_Summary_',f'stage3/Published/schoology/v{self.version}/Cube_Question_Summary_', primary_key='ID')
        # oea.add_to_lake_db(f'stage2/Enriched/schoology/v{self.version}/Cube_Question_Summary_')
        # oea.add_to_lake_db(f'stage3/Published/schoology/v{self.version}/Cube_Question_Summary_')

        # Fact_student_submissions = oea.load(f'stage3/Published/schoology/v{schoology.version}/fact_student_submission')
        # Fact_student_submissions = Fact_student_submissions.withColumnRenamed('Standard', 'Standards')
        Fact_student_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/fact_student_submission')
        SubjectJoin = Fact_student_submissions.select("Item_ID", "Subject").dropDuplicates(["Item_ID", "Subject"])
        Fact_student_submissions = Fact_student_submissions.withColumn(
            "Total_Seconds",
            F.expr(
                "int(split(Total_Time, ':')[0])*3600 + "
                "int(split(Total_Time, ':')[1])*60 + "
                "int(split(Total_Time, ':')[2])"
            )
        )
        Fact_student_submissionsstudent = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_student_hash')
        Fact_student_submissionsstudent = Fact_student_submissionsstudent.select(
            F.col("User_UID"),
            F.col("StudentName_Hash")
        )
        Fact_student_submissions=Fact_student_submissions.join(Fact_student_submissionsstudent,on=['User_UID'],how='left')
        Fact_student_submissions = Fact_student_submissions.withColumnRenamed('Standard', 'Standards')
        Fact_student_submissions = Fact_student_submissions.withColumn("Points_Received", col("Points_Received").cast("double")) \
                                     .withColumn("Points_Possible", col("Points_Possible").cast("double"))
        # unique_subm_pts_received = Fact_student_submissions.dropDuplicates(
        #     ['School_ID','Item_ID','User_UID','Question_ID','Points_Received', 'Points_Possible']
        # )
        Cube_Question_Summary = Fact_student_submissions.groupBy('Item_ID', 'Question_ID').agg(
            F.sum("Points_Possible").alias("Total_Possible_Point"),
            F.sum('Points_Received').alias("Total_Score"),
            (F.sum("Points_Received") / F.sum("Points_Possible")).alias("Grade_Average"),
            ((1 - (F.sum(F.col('Points_Received')) / F.sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers')
        )
        # Filter out submissions for each choice and include Points_Received and Student_Name
        ChoiceSubmissions = Fact_student_submissions.dropDuplicates(["Question_ID", "Answer_Submission", "Position_Number","Points_Received", "User_Name","StudentName_Hash"])        # Total submissions per question
        # Total submissions per question
        TotalSubmissions = ChoiceSubmissions.groupBy("Question_ID","Position_Number").agg(F.count("Answer_Submission").alias("Total_Submissions")) 
        # Calculate the percentage of each choice being selected
        ChoiceSubmissions = ChoiceSubmissions.withColumn("Points_Received", col("Points_Received").cast("double")).withColumn("Points_Possible", col("Points_Possible").cast("double"))
        ChoicePercentage = ChoiceSubmissions.groupBy("Question_ID", "Position_Number","Answer_Submission").agg(
            # F.count("Answer_Submission").alias("Choice_Submission_Count"),
            F.sum(F.when(F.col("Points_Received").cast("double") < F.col("Points_Possible").cast("double"), 1).otherwise(0)).alias("Choice_Submission_Count"), #Incorrect answer option count
            F.min("Points_Received").alias("Points_Received"), # Keep Points_Received for correct/incorrect marking
            F.max("Points_Possible").alias("Max_marks") 
        ).join(TotalSubmissions, on=["Question_ID","Position_Number"]).withColumn(
            "Choice_Percentage", F.col("Choice_Submission_Count") / F.col("Total_Submissions")
        )
        # Mark if a choice is correct or incorrect based on Points_Received
        ChoiceWithCorrectness = ChoicePercentage.withColumn(
            "Is_Correct",
            when(col("Points_Received").cast("double") < col("Max_marks").cast("double"), lit("Incorrect"))  # If Points_Received < Points_Possible, mark as Incorrect
            .otherwise(lit("Correct"))  # Otherwise, it's Correct
        )
        # Filter only incorrect choices
        IncorrectChoices = ChoiceWithCorrectness.filter(F.col("Is_Correct") == "Incorrect")
        # Create a list of student names for each incorrect choice
        StudentChoices = ChoiceSubmissions.filter(col("Points_Received").cast("double") < col("Points_Possible").cast("double")).groupBy("Question_ID","Position_Number","Answer_Submission").agg(
            F.collect_list("User_Name").alias("Students_Who_Chose"),F.collect_list("StudentName_Hash").alias("Students_Who_Chose_Hash")
        )
        # Join the student names back with the incorrect choices
        IncorrectChoicesWithStudents = IncorrectChoices.join(StudentChoices, on=["Question_ID","Position_Number", "Answer_Submission"],how="left")
        # Concatenate results to show percentages for only incorrect choices and student names
        FinalResults = IncorrectChoicesWithStudents.withColumn(
            "Choice_Details", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose) = 1 THEN
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                    ELSE
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                END
            """)
        ).withColumn(
            "Choice_Details_Hash", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose_Hash) = 1 THEN
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                    ELSE
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                END
            """)
        )
        
        
        FinalResults = FinalResults.withColumn(
            "Choice_Details_WithName", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose) = 1 THEN
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            Students_Who_Chose[0], ')'
                        )
                    ELSE
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            CONCAT_WS(', ', Students_Who_Chose), ')'
                        )
                END
            """)
        ).withColumn(
            "Choice_Details_WithName_Hash", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose_Hash) = 1 THEN
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            Students_Who_Chose_Hash[0], ')'
                        )
                    ELSE
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            CONCAT_WS(', ', Students_Who_Chose_Hash), ')'
                        )
                END
            """)
        )
        # Aggregating incorrect choices into a single string per question
        ChoicesPerQuestion = FinalResults.groupBy(
            'Question_ID','Position_Number'
        ).agg(
            F.concat_ws(", ", F.collect_list("Choice_Details")).alias("Incorrect_Choice_Details"),
            F.concat_ws(", ", F.collect_list("Choice_Details_WithName")).alias("Incorrect_Details_Name"),
            F.concat_ws(", ", F.collect_list("Choice_Details_Hash")).alias("Incorrect_Choice_Details_Hash"),
            F.concat_ws(", ", F.collect_list("Choice_Details_WithName_Hash")).alias("Incorrect_Details_Name_Hash")
        ).withColumn(
            "Incorrect_Choice_Details", F.expr("substring(Incorrect_Choice_Details, 1, 7500)")
        ).withColumn(
            "Incorrect_Details_Name", F.expr("substring(Incorrect_Details_Name, 1, 7500)")
        ).withColumn(
            "Incorrect_Choice_Details_Hash", F.expr("substring(Incorrect_Choice_Details_Hash, 1, 7500)")
        ).withColumn(
            "Incorrect_Details_Name_Hash", F.expr("substring(Incorrect_Details_Name_Hash, 1, 7500)")
        )
        # Join with Cube_Question_Summary to include incorrect choice details
        
        DimQuestionData = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_question_data1')
        DimQuestionData = DimQuestionData.drop('Subject')
        DimQuestionData = DimQuestionData.join(
            SubjectJoin,
            on=["Item_ID"],
            how="left"
        )
        
        DimItemName = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_item')
        
        # Join Fact_student_submissions with DimQuestionData on Question_ID and Item_ID
        posNoQues = DimQuestionData.select("Question_ID", "Item_ID", "Position_Number").dropDuplicates(["Question_ID", "Item_ID", "Position_Number"])
        Cube_Question_Summary = Cube_Question_Summary.join(posNoQues,on=["Question_ID", "Item_ID"],how="inner")
        JoinedResults = Cube_Question_Summary.join(ChoicesPerQuestion, on=["Question_ID","Position_Number"], how="left")
        JoinedResults = JoinedResults.join(
            DimQuestionData,
            on=["Question_ID", "Item_ID","Position_Number"],
            how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        JoinedResults = JoinedResults.join(
            DimItemName,
            on=['School_ID', "Item_ID","Item_Name"],
            how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        JoinedResults = JoinedResults.withColumn(
            "Question_no_url",
            trim(regexp_replace(F.col("Question"), r"<[^>]*>", ""))
        )
         
        JoinedResults = JoinedResults.withColumn(
            "Standard",
            F.when(F.col("Standard").isNull() | (F.col("Standard") == "null"), "Other")
            .otherwise(F.col("Standard"))
        )
        JoinedResults = JoinedResults.withColumn(
            "Standards",
            F.when(F.col("Standards").isNull() | (F.col("Standards") == "null"), "Other")
            .otherwise(F.col("Standards"))
        )
        
        Fact_section_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_section_Hash')
        Fact_section_submissions = Fact_section_submissions.select(
            F.col("Section_Instructors"),
            F.col("TeacherName_Hash")
        )
        JoinedResults = JoinedResults.join(Fact_section_submissions,on=['Section_Instructors'],how='left')
        Cube_Question_Summary = JoinedResults.withColumn(
                    "ID",
                    F.sha2(
                        F.concat(
                            F.when(F.col("School_ID").isNotNull(), F.col("School_ID")).otherwise(F.lit("DEFAULT_SCHOOL_ID")),
                            F.when(F.col("Item_ID").isNotNull(), F.col("Item_ID")).otherwise(F.lit("DEFAULT_ITEM_ID")),
                            F.when(F.col("Item_Name").isNotNull(), F.col("Item_Name")).otherwise(F.lit("DEFAULT_ITEM_NAME")),
                            F.when(F.col("Question_ID").isNotNull(), F.col("Question_ID")).otherwise(F.lit("DEFAULT_QUESTION_ID")),
                            F.when(F.col("Position_Number").isNotNull(), F.col("Position_Number")).otherwise(F.lit("DEFAULT_POSITION_NO")),
                            F.when(F.col("Standard").isNotNull(), F.col("Standard")).otherwise(F.lit("DEFAULT_STANDARD")),
                            F.when(F.col("Correct_Answer").isNotNull(), F.col("Correct_Answer")).otherwise(F.lit("DEFAULT_CORRECT_ANSWER"))
                        ), 
                        256
                    )
        )
        params = {
            'req_cols': [
                ['School_ID', 'Question_ID', 'Standard'],
                ['School_ID', 'Question_ID', 'Correct_Answer']
            ]
        }    
        self.delete_stale_rows(Cube_Question_Summary, f'stage2/Enriched/canvas/v{self.version}/Cube_Question_Summary',f'stage3/Published/canvas/v{self.version}/Cube_Question_Summary', params)
        schoology.publish(Cube_Question_Summary, f'stage2/Enriched/canvas/v{schoology.version}/Cube_Question_Summary',f'stage3/Published/canvas/v{schoology.version}/Cube_Question_Summary', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/Cube_Question_Summary')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/Cube_Question_Summary')

        from pyspark.sql.functions import col, sum, countDistinct
        DimQuestionData = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_question_data1')
        Fact_student_submissions = Fact_student_submissions.join(DimQuestionData, on=["Question_ID","Position_Number"], how="left")
        Cube_QuestionIncorrectChoice_Summary = Fact_student_submissions.rollup('Question_ID','UKey','Answer_Submission').agg(
            countDistinct("User_UID").alias("Total_Student"),
            sum("Points_Possible").alias("Total_Possible_Point"),
            sum('Points_Received').alias("Total_Score"),
            (sum("Points_Received") / sum("Points_Possible")).alias("Grade_Average"),
            ((1 - (sum(F.col('Points_Received')) / sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers'),
        )
        
        # Cube_QuestionIncorrectChoice_Summary = Cube_QuestionIncorrectChoice_Summary.withColumn("ID", monotonically_increasing_id())
        Cube_QuestionIncorrectChoice_Summary = Cube_QuestionIncorrectChoice_Summary.withColumn(
            "ID",
            F.sha2(
                F.concat(
                    F.when(F.col("UKey").isNotNull(), F.col("UKey")).otherwise(F.lit("DEFAULT_UKey")),
                    F.when(F.col("Question_ID").isNotNull(), F.col("Question_ID")).otherwise(F.lit("DEFAULT_QUESTION_ID")),
                    F.when(F.col("Answer_Submission").isNotNull(), F.col("Answer_Submission")).otherwise(F.lit("DEFAULT_ANSWER_SUBMISSION"))
                ), 
                256
            )
        )

        self.publish(Cube_QuestionIncorrectChoice_Summary, f'stage2/Enriched/canvas/v{self.version}/Cube_QuestionIncorrectChoice_Summary',f'stage3/Published/canvas/v{self.version}/Cube_QuestionIncorrectChoice_Summary', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/Cube_QuestionIncorrectChoice_Summary')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/Cube_QuestionIncorrectChoice_Summary')
        

        # cube Question Summary Overall

        # Fact_student_submissions = oea.load(f'stage3/Published/schoology/v{schoology.version}/fact_student_submission')
        # DimQuestionData = oea.load(f'stage3/Published/schoology/v{schoology.version}/dim_question_data')
        # DimQuestionData = DimQuestionData.drop("Standards")
        # # DimQuestionData.filter(DimQuestionData['Question_ID']=='2018668896').select('School_ID','Session','Grade','Subject',"Question_ID", "Item_ID","Item_Name","Correct_Answer","Standard",'Qkey','Ukey').show()
        # Fact_student_submissions = Fact_student_submissions.fillna({"Standard": "null"})
        # DimQuestionData = DimQuestionData.fillna({"Standard": "null"})
        # Fact_student_submissions = Fact_student_submissions.drop("Correct_Answer")
        # # DimQuestionData = DimQuestionData.withColumnRenamed('Standard', 'Standards')
        # # Join Fact_student_submissions with DimQuestionData on Question_ID and Item_ID
        # Fact_with_Dim = Fact_student_submissions.join(
        #     DimQuestionData,
        #     on=["Question_ID", "Item_ID","Item_Name","Standard"],
        #     how="left"  # Adjust join type as needed (inner, left, etc.)
        # )
        # # Fact_with_Dim.filter(
        # #     (Fact_with_Dim['Question'].isNull())
        # # ).select('Question_ID', 'Standard').show()

        # # Rollup the data by Item_Name, Question, and Correct_Answer to get summary statistics
        # Cube_Question_Summary = Fact_with_Dim.groupBy('uKey','Subject_ID','Question_No', 'Question', 'Correct_Answer',"Standard").agg(
        #     F.sum("Points_Possible").alias("Total_Possible_Point"),
        #     F.sum('Points_Received').alias("Total_Score"),
        #     (F.sum("Points_Received") / F.sum("Points_Possible")).alias("Grade_Average"),
        #     ((1 - (F.sum(F.col('Points_Received')) / F.sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers')
        # )

        # # Filter out submissions for each choice and include Points_Received and Student_Name
        # ChoiceSubmissions = Fact_with_Dim.select('uKey','Subject_ID','Question_No',"Question", "Answer_Submission", "Points_Received", "User_Name", "Correct_Answer","Standard")

        # # Total submissions per question
        # TotalSubmissions = ChoiceSubmissions.groupBy('uKey','Subject_ID','Question_No', "Question",'Correct_Answer',"Standard").agg(F.count("Answer_Submission").
        # alias("Total_Submissions"))

        # # Calculate the percentage of each choice being selected
        # ChoicePercentage = ChoiceSubmissions.groupBy('uKey','Subject_ID','Question_No' ,"Question", "Answer_Submission","Correct_Answer","Standard").agg(
        #     F.count("Answer_Submission").alias("Choice_Submission_Count"),
        #     F.min("Points_Received").alias("Points_Received")  # Keep Points_Received for correct/incorrect marking
        # ).join(TotalSubmissions, on=['uKey','Subject_ID','Question_No', "Question",'Correct_Answer',"Standard"],how="left").withColumn(
        #     "Choice_Percentage", F.col("Choice_Submission_Count") / F.col("Total_Submissions")
        # )

        # # Mark if a choice is correct or incorrect based on Points_Received
        # ChoiceWithCorrectness = ChoicePercentage.withColumn(
        #     "Is_Correct", F.when(F.col("Points_Received") > 0, F.lit("Correct")).otherwise(F.lit("Incorrect"))
        # )

        # # Filter only incorrect choices
        # IncorrectChoices = ChoiceWithCorrectness.filter(F.col("Is_Correct") == "Incorrect")

        # # Create a list of student names for each incorrect choice
        # StudentChoices = ChoiceSubmissions.filter(F.col("Points_Received") == 0).groupBy('uKey','Subject_ID','Question_No', "Question", "Answer_Submission","Correct_Answer","Standard").agg(
        #     F.collect_list("User_Name").alias("Students_Who_Chose")
        # )
        # # Join the student names back with the incorrect choices
        # IncorrectChoicesWithStudents = IncorrectChoices.join(StudentChoices, on=['uKey','Subject_ID','Question_No', "Question", "Answer_Submission", "Correct_Answer","Standard"],how="left")
        # # Concatenate results to show percentages for only incorrect choices and student names
        # FinalResults = IncorrectChoicesWithStudents.withColumn(
        #     "Choice_Details", F.expr("""
        #         CASE
        #             WHEN size(Students_Who_Chose) = 1 THEN
        #                 CONCAT(
        #                     FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
        #                 )
        #             ELSE
        #                 CONCAT(
        #                     FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
        #                 )
        #         END
        #     """)
        # )
        # FinalResults = FinalResults.withColumn(
        #     "Choice_Details_WithName", F.expr("""
        #         CASE
        #             WHEN size(Students_Who_Chose) = 1 THEN
        #                 CONCAT(
        #                     '  [', Answer_Submission, '] (',
        #                     Students_Who_Chose[0], ')'
        #                 )
        #             ELSE
        #                 CONCAT(
        #                     '  [', Answer_Submission, '] (',
        #                     CONCAT_WS(', ', Students_Who_Chose), ')'
        #                 )
        #         END
        #     """)
        # )

        # # Aggregating incorrect choices into a single string per question
        # ChoicesPerQuestion = FinalResults.groupBy(
        #     'uKey','Subject_ID', 'Question_No', 'Question', 'Correct_Answer', 'Standard'
        # ).agg(
        #     F.concat_ws(", ", F.collect_list("Choice_Details")).alias("Incorrect_Choice_Details"),
        #     F.concat_ws(", ", F.collect_list("Choice_Details_WithName")).alias("Incorrect_Details_Name")
        # ).withColumn(
        #     "Incorrect_Choice_Details", F.expr("substring(Incorrect_Choice_Details, 1, 8000)")
        # ).withColumn(
        #     "Incorrect_Details_Name", F.expr("substring(Incorrect_Details_Name, 1, 8000)")
        # )
        # # Join with Cube_Question_Summary to include incorrect choice details
        # JoinedResults = Cube_Question_Summary.join(ChoicesPerQuestion, on=['uKey','Subject_ID','Question_No' ,"Question", "Correct_Answer","Standard"], how="left")
        # JoinedResults = JoinedResults.withColumnRenamed("Standard", "Standards")
        # # Drop rows where Standards is null
        # # JoinedResults = JoinedResults.filter(JoinedResults["Standards"] != "null")
        # JoinedResults = JoinedResults.withColumn(
        #     "Question_no_url",
        #     trim(regexp_replace(col("Question"), r"<[^>]*>", ""))
        # )
        
        # JoinedResults = JoinedResults.withColumn(
        #     "Standards", 
        #     F.when(F.col("Standards").isNull() | (F.col("Standards") == "null"), "Other")
        #     .otherwise(F.col("Standards"))
        # )
        # all_columns = JoinedResults.columns
        # group_by_columns = ['Subject_ID', 'Question_No', 'Grade_Average']
        # agg_expr = []
        
        # for col in all_columns:
        #     if col in group_by_columns:
        #         continue  
        #     elif col == 'Standards' or col == 'Correct_Answer':
        #         agg_expr.append(F.concat_ws(',', F.collect_set(col)).alias(col))
        #     else:
        #         agg_expr.append(F.first(col).alias(col))
        # JoinedResults = JoinedResults.groupBy(*group_by_columns).agg(*agg_expr)
        # # Cube_Question_Summary_Overall = JoinedResults.withColumn("ID", monotonically_increasing_id())
        # Cube_Question_Summary_Overall = JoinedResults.withColumn(
        #     "ID",
        #     F.sha2(
        #         F.concat(
        #             F.when(F.col("Subject_ID").isNotNull(), F.col("Subject_ID")).otherwise(F.lit("DEFAULT_SUBJECT_ID")),
        #             F.when(F.col("Question_No").isNotNull(), F.col("Question_No")).otherwise(F.lit("DEFAULT_QUESTION_NO")),
        #             F.when(F.col("Question").isNotNull(), F.col("Question")).otherwise(F.lit("DEFAULT_QUESTION")),
        #             F.when(F.col("Correct_Answer").isNotNull(), F.col("Correct_Answer")).otherwise(F.lit("DEFAULT_CORRECT_ANSWER")),
        #             F.when(F.col("Standards").isNotNull(), F.col("Standards")).otherwise(F.lit("DEFAULT_STANDARDS"))
        #         ),
        #         256
        #     )
        # )

        # params = {
        #     'req_cols': [
        #         ['Subject_ID', 'Question', 'Standards'],
        #         ['Subject_ID', 'Question', 'Correct_Answer']
        #     ]
        # }    
        # self.delete_stale_rows(Cube_Question_Summary_Overall, f'stage2/Enriched/schoology/v{self.version}/Cube_Question_Summary_Overall',f'stage3/Published/schoology/v{self.version}/Cube_Question_Summary_Overall', params)
        # self.publish(Cube_Question_Summary_Overall, f'stage2/Enriched/schoology/v{self.version}/Cube_Question_Summary_Overall',f'stage3/Published/schoology/v{self.version}/Cube_Question_Summary_Overall', primary_key='ID')
        # oea.add_to_lake_db(f'stage2/Enriched/schoology/v{self.version}/Cube_Question_Summary_Overall')
        # oea.add_to_lake_db(f'stage3/Published/schoology/v{self.version}/Cube_Question_Summary_Overall')

        Fact_student_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/fact_student_submission')
        Fact_student_submissions = Fact_student_submissions.withColumn(
            "Total_Seconds",
            F.expr(
                "int(split(Total_Time, ':')[0])*3600 + "
                "int(split(Total_Time, ':')[1])*60 + "
                "int(split(Total_Time, ':')[2])"
            )
        )
        Fact_student_submissions = Fact_student_submissions.fillna({'Correct_Answer': 'n/a'})
        Fact_student_submissionsstudent = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_student_hash')
        Fact_student_submissionsstudent = Fact_student_submissionsstudent.select(
            F.col("User_UID"),
            F.col("StudentName_Hash")
        )
        Fact_student_submissions=Fact_student_submissions.join(Fact_student_submissionsstudent,on=['User_UID'],how='left')
        dimSubject = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_subject')

        # Filter Fact_student_submissions to keep only rows that match dimSubject
        # Replace 'subject_id' with the actual join key
        Fact_student_submissions = Fact_student_submissions.join(
            dimSubject,
            on='Subject_ID',  # change this to the correct join key
            how='left'
        )
        DimQuestionData = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_question_data1')
        DimQuestionData = DimQuestionData.fillna({'Correct_Answer': 'n/a'})
        DimQuestionData = DimQuestionData.drop("Standards")
        Fact_student_submissions = Fact_student_submissions.fillna({"Standard": "null"})
        DimQuestionData = DimQuestionData.fillna({"Standard": "null"})
        # unique_subm_pts_received = Fact_student_submissions.dropDuplicates(
        #     ['School_ID','Item_ID','User_UID','Question_ID','Points_Received', 'Points_Possible']
        # )
        # unique_subm_pts_received = unique_subm_pts_received.drop("Correct_Answer","Standard",'Position_Number')
        # DimQuestionData = DimQuestionData.withColumnRenamed('Standard', 'Standards')
        # Join Fact_student_submissions with DimQuestionData on Question_ID and Item_ID
        Fact_student_submissions = Fact_student_submissions.drop("Correct_Answer","Standard")
        Fact_with_Dim = Fact_student_submissions.join(
            DimQuestionData,
            on=["Question_ID", "Item_ID","Item_Name","Position_Number"],
            how="left"  # Adjust join type as needed (inner, left, etc.)
        )

        Fact_with_Dim = Fact_with_Dim.withColumn("Points_Received", col("Points_Received").cast("double")).withColumn("Points_Possible", col("Points_Possible").cast("double"))
        Cube_Question_Summary = Fact_with_Dim.groupBy('uKey','Subject_ID','Question_No', 'Question','Position_Number' ,'Correct_Answer',"Standard").agg(
            F.sum("Points_Possible").alias("Total_Possible_Point"),
            F.sum('Points_Received').alias("Total_Score"),
            (F.sum("Points_Received") / F.sum("Points_Possible")).alias("Grade_Average"),
            ((1 - (F.sum(F.col('Points_Received')) / F.sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers')
        )

        Fact_student_submissions = Fact_student_submissions.drop("Correct_Answer","Standard")
        Fact_with_Dim = Fact_student_submissions.join(
            DimQuestionData,
            on=["Question_ID", "Item_ID","Item_Name","Position_Number"],
            how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        Fact_with_Dim = Fact_with_Dim.dropDuplicates(['uKey','Subject_ID','Question_No',"Question","Position_Number", "Answer_Submission", "Points_Received", "User_Name", "Correct_Answer","Standard"])
        # Filter out submissions for each choice and include Points_Received and Student_Name
        ChoiceSubmissions = Fact_with_Dim.select('uKey','Subject_ID','Question_No',"Question","Position_Number", "Answer_Submission", "Points_Received", "Points_Possible","StudentName_Hash","User_Name", "Correct_Answer","Standard")

        # Total submissions per question
        TotalSubmissions = ChoiceSubmissions.groupBy('uKey','Subject_ID','Question_No', "Question","Position_Number",'Correct_Answer',"Standard").agg(F.count("Answer_Submission").alias("Total_Submissions"))
        # Calculate the percentage of each choice being selected
        ChoiceSubmissions = ChoiceSubmissions.withColumn("Points_Received", col("Points_Received").cast("double")).withColumn("Points_Possible", col("Points_Possible").cast("double"))
        ChoicePercentage = ChoiceSubmissions.groupBy('uKey','Subject_ID','Question_No' ,"Question","Position_Number", "Answer_Submission","Correct_Answer","Standard").agg(
            # F.count("Answer_Submission").alias("Choice_Submission_Count"),
            F.sum(F.when(F.col("Points_Received").cast("double") < F.col("Points_Possible").cast("double"), 1).otherwise(0)).alias("Choice_Submission_Count"), #Incorrect answer option count
            F.min("Points_Received").alias("Points_Received"),  # Keep Points_Received for correct/incorrect marking
            F.max("Points_Possible").alias("Max_marks") 
        ).join(TotalSubmissions, on=['uKey','Subject_ID','Question_No', "Question","Position_Number",'Correct_Answer',"Standard"],how="left").withColumn(
            "Choice_Percentage", F.col("Choice_Submission_Count") / F.col("Total_Submissions")
        )
        # Mark if a choice is correct or incorrect based on Points_Received
        ChoiceWithCorrectness = ChoicePercentage.withColumn(
            "Is_Correct", 
            F.when(
                F.col("Points_Received").cast("double") < col("Max_marks").cast("double"), 
                F.lit("Incorrect")
            ).otherwise(F.lit("Correct"))
        )
        # Filter only incorrect choices
        IncorrectChoices = ChoiceWithCorrectness.filter(F.col("Is_Correct") == "Incorrect")
        # Create a list of student names for each incorrect choice
        StudentChoices = ChoiceSubmissions.filter(F.col("Points_Received").cast("double") < F.col("Points_Possible").cast("double")).groupBy('uKey','Subject_ID','Question_No', "Question","Position_Number", "Answer_Submission","Correct_Answer","Standard").agg(
            F.collect_list("User_Name").alias("Students_Who_Chose"),F.collect_list("StudentName_Hash").alias("Students_Who_Chose_Hash")
        )
        # Join the student names back with the incorrect choices
        IncorrectChoicesWithStudents = IncorrectChoices.join(StudentChoices, on=['uKey','Subject_ID','Question_No', "Question", "Position_Number","Answer_Submission", "Correct_Answer","Standard"],how="left")
        # Concatenate results to show percentages for only incorrect choices and student names
        FinalResults = IncorrectChoicesWithStudents.withColumn(
            "Choice_Details", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose) = 1 THEN
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                    ELSE
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                END
            """)
        ).withColumn(
            "Choice_Details_Hash", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose_Hash) = 1 THEN
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                    ELSE
                        CONCAT(
                            FORMAT_NUMBER(Choice_Percentage * 100, 2), '% chose [', Answer_Submission, ']'
                        )
                END
            """)
        )


        FinalResults = FinalResults.withColumn(
            "Choice_Details_WithName", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose) = 1 THEN
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            Students_Who_Chose[0], ')'
                        )
                    ELSE
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            CONCAT_WS(', ', Students_Who_Chose), ')'
                        )
                END
            """)
        ).withColumn(
            "Choice_Details_WithName_Hash", F.expr("""
                CASE
                    WHEN size(Students_Who_Chose_Hash) = 1 THEN
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            Students_Who_Chose_Hash[0], ')'
                        )
                    ELSE
                        CONCAT(
                            '  [', Answer_Submission, '] (',
                            CONCAT_WS(', ', Students_Who_Chose_Hash), ')'
                        )
                END
            """)
        )
        # Aggregating incorrect choices into a single string per question
        ChoicesPerQuestion = FinalResults.groupBy(
            'uKey','Subject_ID', 'Question_No', 'Question',"Position_Number", 'Correct_Answer', 'Standard'
        ).agg(
            F.concat_ws(", ", F.collect_list("Choice_Details")).alias("Incorrect_Choice_Details"),
            F.concat_ws(", ", F.collect_list("Choice_Details_WithName")).alias("Incorrect_Details_Name"),
            F.concat_ws(", ", F.collect_list("Choice_Details_Hash")).alias("Incorrect_Choice_Details_Hash"),
            F.concat_ws(", ", F.collect_list("Choice_Details_WithName_Hash")).alias("Incorrect_Details_Name_Hash")
        ).withColumn(
            "Incorrect_Choice_Details", F.expr("substring(Incorrect_Choice_Details, 1, 7500)")
        ).withColumn(
            "Incorrect_Details_Name", F.expr("substring(Incorrect_Details_Name, 1, 7500)")
        ).withColumn(
            "Incorrect_Choice_Details_Hash", F.expr("substring(Incorrect_Choice_Details_Hash, 1, 7500)")
        ).withColumn(
            "Incorrect_Details_Name_Hash", F.expr("substring(Incorrect_Details_Name_Hash, 1, 7500)")
        )
        # Join with Cube_Question_Summary to include incorrect choice details
        JoinedResults = Cube_Question_Summary.join(ChoicesPerQuestion, on=['uKey','Subject_ID','Question_No' ,"Question", "Position_Number","Correct_Answer","Standard"], how="left")
        JoinedResults = JoinedResults.withColumnRenamed("Standard", "Standards")
        # Drop rows where Standards is null
        # JoinedResults = JoinedResults.filter(JoinedResults["Standards"] != "null")
        JoinedResults = JoinedResults.withColumn(
            "Question_no_url",
            trim(regexp_replace(F.col("Question"), r"<[^>]*>", ""))
        )

        JoinedResults = JoinedResults.withColumn(
            "Standards",
            F.when(F.col("Standards").isNull() | (F.col("Standards") == "null"), "Other")
            .otherwise(F.col("Standards"))
        )
        standard = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_standard')
        # standard = standard.withColumnRenamed("Schoology_Standard", "Standards")
        desc_stand = standard[["Description","Schoology_Standard"]].dropDuplicates()
        JoinedResults = JoinedResults.join(
            desc_stand,
            JoinedResults.Standards.contains(desc_stand.Schoology_Standard),
            how="left"
        )
        JoinedResults =JoinedResults.drop("Schoology_Standard")
        JoinedResults = JoinedResults.dropDuplicates(['Subject_ID','Question_No',"Position_Number",'Correct_Answer','Standards'])
        JoinedResults = JoinedResults.withColumn("Incorrect_Choice_Details", expr("SUBSTRING(Incorrect_Choice_Details, 1, 7500)"))
        JoinedResults = JoinedResults.withColumn("Incorrect_Details_Name", expr("SUBSTRING(Incorrect_Details_Name, 1, 7500)"))
        DimItemName = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_item')
        DimItemName= DimItemName.select(
            F.col("Subject_ID"),
            F.col("Section_Instructors")
        )
        JoinedResults = JoinedResults.join(DimItemName,on=['Subject_ID'],how='left')
        dim_section_hash = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_section_Hash')
        dim_section_hash= dim_section_hash.select(
            F.col("Section_Instructors"),
            F.col("TeacherName_Hash")
        )
        JoinedResults = JoinedResults.join(dim_section_hash,on=['Section_Instructors'],how='left')
        Cube_Question_Summary_Overall = JoinedResults.withColumn(
            "ID",
            F.sha2(
                F.concat_ws(", ",
                    F.when(F.col("Subject_ID").isNotNull(), F.col("Subject_ID")).otherwise(F.lit("DEFAULT_SUBJECT_ID")),
                    F.when(F.col("Question_No").isNotNull(), F.col("Question_No")).otherwise(F.lit("DEFAULT_QUESTION_NO")),
                    # F.when(F.col("Question").isNotNull(), F.col("Question")).otherwise(F.lit("DEFAULT_QUESTION")),
                    F.when(F.col("Position_Number").isNotNull(), F.col("Position_Number")).otherwise(F.lit("DEFAULT_POSITION_NO")),
                    F.when(F.col("Correct_Answer").isNotNull(), F.col("Correct_Answer")).otherwise(F.lit("DEFAULT_CORRECT_ANSWER")),
                    F.when(F.col("Standards").isNotNull(), F.col("Standards")).otherwise(F.lit("DEFAULT_STANDARDS")),
                    F.when(F.col("uKey").isNotNull(), F.col("Standards")).otherwise(F.lit("DEFAULT_STANDARDS"))
                ),
                256
            )
        )

        params = {
            'req_cols': [
                ['Subject_ID', 'Question_No', 'Standards'],
                ['Subject_ID', 'Question_No', 'Correct_Answer']
            ]
        }
        schoology.delete_stale_rows(Cube_Question_Summary_Overall, f'stage2/Enriched/canvas/v{schoology.version}/Cube_Question_Summary_Overall',f'stage3/Published/canvas/v{schoology.version}/Cube_Question_Summary_Overall', params)
        schoology.publish(Cube_Question_Summary_Overall, f'stage2/Enriched/canvas/v{schoology.version}/Cube_Question_Summary_Overall',f'stage3/Published/canvas/v{schoology.version}/Cube_Question_Summary_Overall', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/Cube_Question_Summary_Overall')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/Cube_Question_Summary_Overall')


        # cube User Summary
        Fact_student_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/fact_student_submission')
        Fact_student_submissions = Fact_student_submissions.withColumn(
            "Total_Seconds",
            F.expr(
                "int(split(Total_Time, ':')[0])*3600 + "
                "int(split(Total_Time, ':')[1])*60 + "
                "int(split(Total_Time, ':')[2])"
            )
        )
        # Define window: partition per student-question, order by latest Submission, then longest Total_Time
        window_spec = Window.partitionBy(
            'Section_NID', 'Session', 'Grade', 'Subject',
            'Assessment_type', 'School_ID', 'User_UID', 'Item_ID', 'Question_ID'
        ).orderBy(F.desc('Submission'), F.desc('Total_Seconds'))
        
        # Keep only the top-ranked record
        Fact_student_submissions = (
            Fact_student_submissions
            .withColumn('rn', F.row_number().over(window_spec))
            .filter(F.col('rn') == 1)
            .drop('rn', 'Total_Seconds')  # optional cleanup
        )
        
        Fact_student_submissions = Fact_student_submissions.withColumnRenamed('Standard', 'Standards')
        #  Load data for DimQuestionData
        DimQuestionData = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_question_data1')
        DimQuestionData = DimQuestionData.withColumnRenamed("Standards","Standards_ques")
        DimItem = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_item')
        # Join Fact_student_submissions with DimQuestionData on Question_ID and Item_ID
        Fact_with_Dim = Fact_student_submissions.join(
        DimQuestionData,
        on=['School_ID','Session','Grade','Subject',"Question_ID", "Item_ID","Item_Name","Identifier","Assessment_type"],
        how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        Fact_with_Dim =Fact_with_Dim.drop("Standards_ques")
        Fact_with_Dim = Fact_with_Dim.join(
        DimItem,
        on=['School_ID',"Item_ID",'Item_Name','Subject_ID'],
        how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        Fact_with_Dim = Fact_with_Dim.dropDuplicates(
        ['Section_NID', 'Section_Instructors', 'Session', 'Grade', 'Subject', 'Assessment_type',
        'User_UID', 'User_Name', 'Item_ID', 'Item_Name', 'Question_ID', 'Question_No']
        )
        # Total Possible Ponts from Question table for each assessment
        distinct_question_count = Fact_with_Dim.groupby('Item_ID').agg(
            F.countDistinct('Question_ID').alias('Ass_Total_Points')
        )
        Fact_with_Dim = Fact_with_Dim.join(
            distinct_question_count,
            on='Item_ID',
            how='left'  
        )
        Fact_with_Dim = Fact_with_Dim.withColumn(
            'Total_Possible_Point', F.col('Ass_Total_Points')
        )
        Fact_with_Dim = Fact_with_Dim.drop('Ass_Total_Points')
        Fact_with_Dim = Fact_with_Dim.withColumn(
            'Question_No',F.col('Question_No').cast('int')
        )

        Cube_OverallPerformance_Summary = Fact_with_Dim.groupby('Identifier','Item_ID','Item_Name','Question_ID','Question_No','Standards').agg(
            sum("Points_Possible").alias("Total_Possible_Point"),
            sum('Points_Received').alias("Total_Score"),
            (sum("Points_Received") / sum("Points_Possible")).alias("Grade_Average"),
            ((1 - (sum(F.col('Points_Received')) / sum(F.col('Points_Possible'))))).alias('Percentage_InCorrect_Answers'),
        )
        # Cube_OverallPerformance_Summary = Cube_OverallPerformance_Summary.withColumn("ID", monotonically_increasing_id())
        Cube_OverallPerformance_Summary = Cube_OverallPerformance_Summary.withColumn(
            "ID", 
            F.sha2(
                F.concat_ws(", ",
                    F.when(F.col("Identifier").isNotNull(), F.col("Identifier")).otherwise(F.lit("DEFAULT_IDENTIFIER")),
                    F.when(F.col("Item_ID").isNotNull(), F.col("Item_ID")).otherwise(F.lit("DEFAULT_ITEM_ID")),
                    F.when(F.col("Item_Name").isNotNull(), F.col("Item_Name")).otherwise(F.lit("DEFAULT_ITEM_NAME")),
                    F.when(F.col("Question_ID").isNotNull(), F.col("Question_ID")).otherwise(F.lit("DEFAULT_QUESTION_ID")),
                    F.when(F.col("Question_No").isNotNull(), F.col("Question_No")).otherwise(F.lit("DEFAULT_QUESTION_NO")),
                    F.when(F.col("Standards").isNotNull(), F.col("Standards")).otherwise(F.lit("DEFAULT_STANDARDS"))
                ), 
                256
            )
        )

        params = {'req_cols': [['Item_ID','Question_ID', 'Standards']]}
        self.delete_stale_rows(Cube_OverallPerformance_Summary, f'stage2/Enriched/canvas/v{self.version}/Cube_OverallPerformance_Summary',f'stage3/Published/canvas/v{self.version}/Cube_OverallPerformance_Summary',params)
        self.publish(Cube_OverallPerformance_Summary, f'stage2/Enriched/canvas/v{self.version}/Cube_OverallPerformance_Summary',f'stage3/Published/canvas/v{self.version}/Cube_OverallPerformance_Summary', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{self.version}/Cube_OverallPerformance_Summary')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{self.version}/Cube_OverallPerformance_Summary')

        # # cube User Summary - Paginated
        Fact_student_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/fact_student_submission')
        # Convert Total_Time (hh:mm:ss) → seconds for numeric sorting
        Fact_student_submissions = Fact_student_submissions.withColumn(
            "Total_Seconds",
            F.expr(
                "int(split(Total_Time, ':')[0])*3600 + "
                "int(split(Total_Time, ':')[1])*60 + "
                "int(split(Total_Time, ':')[2])"
            )
        )
        
        # Define window: partition per student-question, order by latest Submission, then longest Total_Time
        window_spec = Window.partitionBy(
            'Section_NID', 'Session', 'Grade', 'Subject',
            'Assessment_type', 'School_ID', 'User_UID', 'Item_ID', 'Question_ID'
        ).orderBy(F.when(F.col('Submission_Grade') == 'Pending Review', 0).otherwise(1).desc(),F.desc('Submission'), F.desc('Total_Seconds'))
        
        # Keep only the top-ranked record
        Fact_student_submissions = (
            Fact_student_submissions
            .withColumn('rn', F.row_number().over(window_spec))
            .filter(F.col('rn') == 1)
            .drop('rn', 'Total_Seconds')  # optional cleanup
        )
        
        Fact_student_submissions = (
            Fact_student_submissions
            .groupBy(
                'Section_NID', 'Session', 'Grade', 'Subject',
                'Assessment_type', 'School_ID','Subject_ID', 'User_UID', 'User_Name',
                'Item_ID', 'Item_Name', 'Question_ID', 'Standard','Submission'
            )
            .agg(
                F.max('Points_Received').alias('Points_Received'),
                F.max('Points_Possible').alias('Points_Possible')
            )
        )
        Fact_student_submissions = Fact_student_submissions.withColumnRenamed('Standard', 'Standards')
        #  Load data for DimQuestionData
        DimQuestionData = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_question_data1')
        DimQuestionData = DimQuestionData.drop('subject')
        Fact_student_submissions = Fact_student_submissions.withColumn(
            "Standards",
            F.when(F.col("Standards").isNull(), "Other").otherwise(F.col("Standards"))
        )
        DimQuestionData = DimQuestionData.withColumn(
            "Standard",
            F.when(F.col("Standard").isNull(), "Other").otherwise(F.col("Standard"))
        )
        DimQuestionData = DimQuestionData.withColumnRenamed("Standard","Standards_ques")
        DimItem = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_item')
        # Join Fact_student_submissions with DimQuestionData on Question_ID and Item_ID
        Fact_with_SingleQuestion = Fact_student_submissions.join(
        DimQuestionData,
        on=['School_ID','Session','Grade',"Question_ID", "Item_ID","Item_Name","Assessment_type"],
        how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        Fact_with_Dim = Fact_student_submissions.join(
        DimQuestionData,
        on=['School_ID','Session','Grade',"Question_ID", "Item_ID","Item_Name","Assessment_type"],
        how="right"  # Adjust join type as needed (inner, left, etc.)
        )
        Cube_User_Summary_Filter = Fact_with_Dim.groupby('Standards_ques','Item_Name','Session').agg(
        sum("Points_Possible").alias("Total_Possible_Point_standard")
        )
        Fact_with_Dim = Fact_with_Dim.join(Cube_User_Summary_Filter,on=['Standards_ques','Item_Name','Session'],how='left')
        Fact_with_Dim = Fact_with_Dim.filter(
            (F.col('Total_Possible_Point_standard') > 0) & (F.col('Total_Possible_Point_standard').isNotNull())
        )
        Fact_with_SingleQuestion =Fact_with_SingleQuestion.drop("Standards_ques")
        Fact_with_Dim = Fact_with_Dim.join(
        DimItem,
        on=['School_ID',"Item_ID",'Item_Name','Subject_ID'],
        how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        Fact_with_SingleQuestion = Fact_with_SingleQuestion.join(
        DimItem,
        on=['School_ID',"Item_ID",'Item_Name','Subject_ID'],
        how="left"  # Adjust join type as needed (inner, left, etc.)
        )
        Fact_with_SingleQuestion = Fact_with_SingleQuestion.withColumnRenamed("Standards","Standards_ques")
        Fact_with_SingleQuestion = Fact_with_SingleQuestion.dropDuplicates(
        ['Section_NID', 'Section_Instructors', 'Session', 'Grade', 'Subject', 'Assessment_type',
        'User_UID', 'User_Name', 'Item_ID', 'Item_Name', 'Question_ID', 'Question_No']
        )
        Fact_with_Dim = Fact_with_Dim.dropDuplicates(
        ['Section_NID', 'Section_Instructors', 'Session', 'Grade', 'Subject', 'Assessment_type',
        'User_UID', 'User_Name', 'Item_ID', 'Item_Name', 'Question_ID', 'Question_No','Standards_ques']
        )
        
        Cube_User_Question_Summary = Fact_with_SingleQuestion.groupby('Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','User_UID','User_Name','Item_ID','Item_Name','School_ID').agg(
        round(sum("Total_Points"),2).alias("Max_Total_Possible_Point_By_Question"),
        round(sum('Points_Received'),2).alias("Total_Score_By_Question")
        )
        max_values = Cube_User_Question_Summary.groupby(
            'Section_NID', 'Section_Instructors', 'Session', 'Grade', 'Subject',
            'Assessment_type', 'Item_ID', 'Item_Name','School_ID'
        ).agg(
            F.max("Max_Total_Possible_Point_By_Question").alias("Total_Possible_Point_By_Question")
        )
        
        # Step 3: Join the maximum values back to the original DataFrame
        Cube_User_Question_Summary = Cube_User_Question_Summary.join(
            max_values,
            on=['Section_NID', 'Section_Instructors', 'Session', 'Grade', 'Subject',
                'Assessment_type', 'Item_ID', 'Item_Name','School_ID'],
            how="left"
        )
        Cube_User_Overall_Summary = Cube_User_Question_Summary.groupby('Session','Grade','Subject','Assessment_type','Item_Name','School_ID').agg(
        round(sum("Total_Possible_Point_By_Question"),2).alias("Total_Possible_Point_By_Overall"),
        round(sum('Total_Score_By_Question'),2).alias("Total_Score_By_Overall")
        )
        
        Cube_User_OverallYear_Summary = Cube_User_Question_Summary.groupby('Session','Grade','Subject','Assessment_type','School_ID').agg(
        round(sum("Total_Possible_Point_By_Question"),2).alias("Total_Possible_Point_By_OverallYear"),
        round(sum('Total_Score_By_Question'),2).alias("Total_Score_By_OverallYear")
        )
        Cube_UserCount_Item_Summary = Fact_with_Dim.groupby('Session','Grade','Subject','Assessment_type','Item_Name','School_ID').agg(
        countDistinct('User_UID').alias("countStudent")
        )  
        Cube_User_Item_Summary = Fact_with_Dim.groupby('Session','Grade','Subject','Assessment_type','Item_Name','Question_No','Standards_ques','School_ID').agg(
                round(sum('Points_Received'),2).alias("Total_Score_By_Item"),
                F.max('Total_Points').alias("Max_Possible_Point_By_Item")
                )
        Cube_User_Item_Summary=Cube_User_Item_Summary.join(Cube_UserCount_Item_Summary,on=['Session','Grade','Subject','Assessment_type','Item_Name','School_ID'],how='left')        
        Cube_User_Item_Summary = Cube_User_Item_Summary.withColumn(
            "Total_Possible_Point_By_Item",
            F.col("countStudent") * F.col("Max_Possible_Point_By_Item")
        )
        
        Cube_User_Item_Summary = Cube_User_Item_Summary.drop("countStudent", "Max_Possible_Point_By_Item")  
        Cube_User_Standard_Summary = Fact_with_Dim.groupby('Session','Grade','Subject','Assessment_type','Item_Name','Standards_ques','School_ID').agg(
        round(sum("Total_Points"),2).alias("Total_Possible_Point_By_Standard"),
        round(sum('Points_Received'),2).alias("Total_Score_By_Standard")
        )
        Cube_User_Section_Summary = Cube_User_Question_Summary.groupby('Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','Item_ID','Item_Name','School_ID').agg(
        round(sum("Total_Possible_Point_By_Question"),2).alias("Total_Possible_Point_By_Section"),
        round(sum('Total_Score_By_Question'),2).alias("Total_Score_By_Section")
        )
        Cube_User_Summary = Fact_with_Dim.groupby('Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','User_UID','User_Name','Item_ID','Item_Name','Question_ID','Question_No','Standards_ques','School_ID').agg(
        round(sum("Total_Points"),2).alias("Total_Possible_Point"),
        round(sum('Points_Received'),2).alias("Total_Score")
        )
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_Overall_Summary,on=['Session','Grade','Subject','Assessment_type','Item_Name','School_ID'],how='left')
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_OverallYear_Summary,on=['Session','Grade','Subject','Assessment_type','School_ID'],how='left')
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_Item_Summary,on=['Session','Grade','Subject','Assessment_type','Item_Name','Question_No','Standards_ques','School_ID'],how='left')
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_Standard_Summary,on=['Session','Grade','Subject','Assessment_type','Item_Name','Standards_ques','School_ID'],how='left')
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_Section_Summary,on=['Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','Item_ID','Item_Name','School_ID'],how='left')
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_Question_Summary,on=['Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','User_UID','User_Name','Item_ID','Item_Name','School_ID'],how='left')
        Cube_User_SummaryUserPossiblePoint=Cube_User_Summary.groupby('Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','User_UID','User_Name','School_ID','Standards_ques').agg(round(sum("Total_Possible_Point"),2).alias("Total_Possible_Point_User"))
        Cube_User_SummaryUserPossiblePoint=Cube_User_SummaryUserPossiblePoint.groupby('Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','School_ID','Standards_ques').agg(max("Total_Possible_Point_User").alias("Total_Possible_Point_Useryear"))
        Cube_User_SummaryUseroverallPossiblePoint=Cube_User_SummaryUserPossiblePoint.groupby('Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','School_ID',).agg(sum("Total_Possible_Point_Useryear").alias("Total_Possible_Point_Useroverallyear"))
        
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_SummaryUserPossiblePoint,on=['Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','School_ID','Standards_ques'],how='left')
        Cube_User_Summary=Cube_User_Summary.join(Cube_User_SummaryUseroverallPossiblePoint,on=['Section_NID','Section_Instructors','Session','Grade','Subject','Assessment_type','School_ID'],how='left')
        # Cube_User_Summary.count()
        Fact_student_submissionsstudent = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_student_hash')
        Fact_student_submissionsstudent = Fact_student_submissionsstudent.select(
            F.col("User_UID"),
            F.col("StudentName_Hash")
        )
        Cube_User_Summary=Cube_User_Summary.join(Fact_student_submissionsstudent,on=['User_UID'],how='left')
        Fact_section_submissions = oea.load(f'stage3/Published/canvas/v{schoology.version}/dim_section_Hash')
        Fact_section_submissions = Fact_section_submissions.select(
            F.col("Section_NID"),
            F.col("TeacherName_Hash")
        )
        Cube_User_Summary = Cube_User_Summary.join(Fact_section_submissions,on=['Section_NID'],how='left')
        # Cube_User_Summary = Cube_User_Summary.withColumn("ID", monotonically_increasing_id())
        
        Cube_User_Summary = Cube_User_Summary.withColumn(
            "ID",
            F.sha2(
                F.concat_ws(", ",
                    F.when(F.col("Section_NID").isNotNull(), F.col("Section_NID")).otherwise(F.lit("DEFAULT_SECTION_NID")),
                    F.when(F.col("Session").isNotNull(), F.col("Session")).otherwise(F.lit("DEFAULT_SESSION")),
                    F.when(F.col("Assessment_type").isNotNull(), F.col("Assessment_type")).otherwise(F.lit("DEFAULT_ASSESSMENT_TYPE")),
                    F.when(F.col("User_UID").isNotNull(), F.col("User_UID")).otherwise(F.lit("DEFAULT_USER_UID")),
                    F.when(F.col("User_Name").isNotNull(), F.col("User_Name")).otherwise(F.lit("DEFAULT_USER_NAME")),
                    F.when(F.col("Item_ID").isNotNull(), F.col("Item_ID")).otherwise(F.lit("DEFAULT_ITEM_ID")),
                    F.when(F.col("Question_ID").isNotNull(), F.col("Question_ID")).otherwise(F.lit("DEFAULT_QUESTION_ID")),
                    F.when(F.col("Question_No").isNotNull(), F.col("Question_No")).otherwise(F.lit("DEFAULT_QUESTION_NO")),
                    F.when(F.col("Standards_ques").isNotNull(), F.col("Standards_ques")).otherwise(F.lit("DEFAULT_STANDARDS_QUES")),
                    F.when(F.col("School_ID").isNotNull(), F.col("School_ID")).otherwise(F.lit("DEFAULT_SCHOOL_ID"))
                ),
                256
            )
        )
        Cube_User_Summary=Cube_User_Summary.withColumnRenamed('Standards_ques', 'Standards')
        params = {
            'req_cols': [
                ['School_ID', 'Question_ID', 'Standards']
            ]
        }
        Cube_User_Summary = Cube_User_Summary.dropDuplicates(["ID"])
        
        # print(Cube_User_Summary.count())
        schoology.delete_stale_rows(Cube_User_Summary, f'stage2/Enriched/canvas/v{schoology.version}/Cube_User_Summary',f'stage3/Published/canvas/v{schoology.version}/Cube_User_Summary',params)
        schoology.publish(Cube_User_Summary, f'stage2/Enriched/canvas/v{schoology.version}/Cube_User_Summary',f'stage3/Published/canvas/v{schoology.version}/Cube_User_Summary', primary_key='ID')
        oea.add_to_lake_db(f'stage2/Enriched/canvas/v{schoology.version}/Cube_User_Summary')
        oea.add_to_lake_db(f'stage3/Published/canvas/v{schoology.version}/Cube_User_Summary')



        

schoology = Schoology()

In [ ]:
schoology.ingest_schoology_dataset("canvas/v0.1")